# Signature based kNN anomaly detection pipeline.

## 0. Loading and initial setup

In [ ]:
%%capture
from IPython.display import display
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import iisignature
import faiss

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))

RESULTS = PROJECT / 'results'
LABELS = PROJECT / 'data' / 'labels'
BENCHMARKS = PROJECT / 'benchmarks'

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

CONFIG = __import__('yaml').safe_load(
    (PROJECT / 'config' / 'config.yaml').read_text(encoding='utf-8'))

TOY_SEED = 42
TOY_STREAMS = 10_000
TOY_LENGTH = 100
CORPUS_NOISE = 0.2
TOY_TRUNC = 5

toy_time = np.linspace(0, 4 * np.pi, TOY_LENGTH)
toy_signal = np.sin(toy_time)
toy_corpus = (toy_signal[None, :]
              + np.random.default_rng(TOY_SEED).normal(
                  0, CORPUS_NOISE, size=(TOY_STREAMS, TOY_LENGTH)))

CMAPSS = PROJECT / 'CMAPSS'
SUBSETS = ('FD001', 'FD002', 'FD003', 'FD004')
REGIMES = {'FD001': '1 condition, 1 fault', 'FD002': '6 conditions, 1 fault',
           'FD003': '1 condition, 2 faults', 'FD004': '6 conditions, 2 faults'}

SETTINGS = ['setting{0}'.format(i) for i in (1, 2, 3)]
SENSORS = ['sensor{0}'.format(i) for i in range(1, 22)]
CHANNELS = SETTINGS + SENSORS
COLUMNS = ['unit', 'cycle'] + CHANNELS

HEALTH_HORIZON = 30
CORPUS_RUL_FLOOR = 50

def read_cmapss(kind, subset):
    frame = pd.read_csv(CMAPSS / '{0}_{1}.txt'.format(kind, subset),
                        sep=r'\s+', header=None, engine='python')
    frame = frame.dropna(axis=1, how='all')
    frame.columns = COLUMNS
    return frame

def load_cmapss(subset):
    to_failure = read_cmapss('train', subset)
    truncated = read_cmapss('test', subset)

    final = pd.read_csv(CMAPSS / 'RUL_{0}.txt'.format(subset),
                        sep=r'\s+', header=None, engine='python').iloc[:, 0]
    final.index = range(1, len(final) + 1)

    to_failure['rul'] = to_failure.groupby('unit').cycle.transform('max') - to_failure.cycle
    truncated['rul'] = (truncated.unit.map(final)
                        + truncated.groupby('unit').cycle.transform('max') - truncated.cycle)

    for frame in (truncated, to_failure):
        frame['degraded'] = frame.rul <= HEALTH_HORIZON
        frame['healthy'] = frame.rul > CORPUS_RUL_FLOOR

    return truncated, to_failure

CMAPSS_DATA = {subset: load_cmapss(subset) for subset in SUBSETS}

EXATHLON = PROJECT / 'data' / 'exathlon'
EXATHLON_CSV = EXATHLON / 'csv'

exathlon_traces = pd.DataFrame([
    {'trace': path.stem, 'side': side, 'path': path,
     'app_id': int(path.stem.split('_')[0]),
     'type_id': int(path.stem.split('_')[1]),
     'input_rate': int(path.stem.split('_')[2]),
     'trace_id': int(path.stem.split('_')[3]),
     'bytes': path.stat().st_size}
    for side in ('corpus', 'test')
    for path in sorted((EXATHLON_CSV / side).glob('*.csv'))
]).sort_values('trace_id').reset_index(drop=True)

exathlon_labels = pd.read_parquet(LABELS / 'exathlon_labels.parquet')
exathlon_anomalies = pd.read_csv(LABELS / 'exathlon_anomalies.csv')

def read_exathlon(trace, columns=None):
    path = exathlon_traces.set_index('trace').loc[trace, 'path']
    return pd.read_csv(path, usecols=columns)

REPORTED = {subset: 'cmapss_{0}_maha'.format(subset.lower()) for subset in SUBSETS}
CURRENT_KEYS = ('roc_auc', 'ad1_f1', 'throughput')

SUMMARIES, run_status = {}, []
for subset, dataset in REPORTED.items():
    path = RESULTS / '{0}_summary.json'.format(dataset)
    summary = json.loads(path.read_text(encoding='utf-8')) if path.exists() else None
    if summary is None:
        state = 'not run'
    elif all(key in summary for key in CURRENT_KEYS):
        state, SUMMARIES[subset] = 'current', summary
    else:
        state = 'stale'
    run_status.append({
        'subset': subset, 'dataset': dataset, 'state': state,
        'written': (pd.Timestamp(path.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')
                    if path.exists() else '-')})
run_status = pd.DataFrame(run_status)

def summarise(**values):
    return pd.DataFrame({'value': pd.Series(values, dtype=object)})


toy_summary = summarise(streams=TOY_STREAMS, points=TOY_LENGTH, noise=CORPUS_NOISE,
                        truncation=TOY_TRUNC)

cmapss_summary = pd.DataFrame([
    {'subset': subset, 'regime': REGIMES[subset],
     'corpus_units': corpus.unit.nunique(), 'corpus_rows': len(corpus),
     'eval_units': evaluation.unit.nunique(), 'eval_rows': len(evaluation),
     'degraded': evaluation.degraded.mean()}
    for subset, (corpus, evaluation) in CMAPSS_DATA.items()])

exathlon_summary = pd.DataFrame([
    {'side': side, 'traces': len(group), 'gb_on_disk': group.bytes.sum() / 1e9,
     'traces_with_anomaly': int(
         exathlon_labels[exathlon_labels.stream.isin(group.trace)]
         .anomalous.map(len).gt(0).sum())}
    for side, group in exathlon_traces.groupby('side')])

exathlon_anomaly_summary = (exathlon_anomalies.groupby('anomaly_type').size()
                            .rename('instances').to_frame())


In [ ]:
%%capture
DETECTOR_SUFFIXES = {
    'maha': 'mahalanobis', 'ifw': 'forest (whitened)', 'ifr': 'forest (raw)',
    'if_whitened': 'forest (whitened)', 'if_raw': 'forest (raw)',
}

COMPARE_COLUMNS = [
    ('f1', 'F1', '{0:.3f}'),
    ('adjusted_f1', 'adjF1', '{0:.3f}'),
    ('roc_auc', 'ROC', '{0:.3f}'),
    ('pr_auc', 'PR-AUC', '{0:.3f}'),
    ('operating_point_auc', 'opAUC', '{0:.3f}'),
    ('ad1_f1', 'AD1', '{0:.3f}'),
    ('ad4_f1', 'AD4', '{0:.3f}'),
]

COMPARE_RATES = [
    ('points_scored_per_second', 'pts/s', '{0:,.0f}'),
    ('peak_rss_mb', 'RSS MB', '{0:,.0f}'),
]

def detector_of(dataset):
    for suffix, name in sorted(DETECTOR_SUFFIXES.items(), key=lambda kv: -len(kv[0])):
        if dataset.endswith('_' + suffix):
            return name
    return '?'

def compare_runs(datasets=None, results_dir=RESULTS):
    datasets = list(datasets if datasets is not None else REPORTED.values())

    rows = []
    for dataset in datasets:
        path = results_dir / '{0}_summary.json'.format(dataset)
        if not path.exists():
            rows.append({'dataset': dataset, 'detector': detector_of(dataset),
                         'state': 'not run', 'written': '-'})
            continue

        metrics = json.loads(path.read_text(encoding='utf-8'))
        rates = (metrics.get('throughput') or {}).get('rates', {})
        counts = (metrics.get('throughput') or {}).get('counts', {})

        row = {
            'dataset': dataset,
            'detector': detector_of(dataset),
            'state': 'current' if all(k in metrics for k in CURRENT_KEYS) else 'STALE',
            'written': pd.Timestamp(path.stat().st_mtime, unit='s').strftime('%m-%d %H:%M'),
            'terms': counts.get('signature_terms'),
            'rank': counts.get('retained_rank'),
        }
        for key, label, _ in COMPARE_COLUMNS:
            row[label] = metrics.get(key)
        for key, label, _ in COMPARE_RATES:
            row[label] = rates.get(key)
        rows.append(row)

    return pd.DataFrame(rows)

def show_comparison(frame):
    formats = dict([(label, spec) for _, label, spec in COMPARE_COLUMNS]
                   + [(label, spec) for _, label, spec in COMPARE_RATES]
                   + [('terms', '{0:,.0f}'), ('rank', '{0:,.0f}')])

    shown = frame.copy()
    for column, spec in formats.items():
        if column in shown.columns:
            shown[column] = [spec.format(v) if pd.notna(v) else '-' for v in shown[column]]
    display(shown)

runs = compare_runs()
show_comparison(runs)

stale = runs[runs.state != 'current']
if len(stale):
    display(pd.DataFrame({'refresh command': [
        'python score_streams.py --dataset {0}'.format(d) for d in stale.dataset]}))

## 1. The toy dataset

We utilise a corpus of noisy sinewaves and squarewaves, to first demonstrate the detector capabilities.

In [ ]:
%%capture
from anomalies_scale.canonical_streams import to_canonical
from anomalies_scale.signature_computer import compute_corpus
from anomalies_scale.covariance_creation import create_covariance, load_whitening
from anomalies_scale.crossvalidated_thresholding import crossvalidated_thresholds
from anomalies_scale.index_creation import build_index, load_index
from anomalies_scale.stream_scoring import score_streams

SCRATCH = PROJECT / 'data' / 'notebook'
SCRATCH.mkdir(parents=True, exist_ok=True)

TOY_GRANULARITY = 5
TOY_CORPUS_N = 500

NEIGHBOURS = CONFIG['detect']['neighbours']
TOY_NEIGHBOURS = 1

TOY_VARIANCE_KEEP = None
TOY_SIG_TOL = 0
STATISTIC = CONFIG['calibrate']['statistic']
FOLDS = CONFIG['calibrate']['folds']
BAND = CONFIG['detect']['band']
VARIANCE_KEEP = CONFIG['metric']['variance_keep']

t = toy_time

def square_wave(x):
    return np.where(np.sin(x) >= 0, 1.0, -1.0)

square_base = square_wave(t)
square_corpus = square_base[None, :] + np.random.default_rng(43).normal(
    0, CORPUS_NOISE, size=(TOY_STREAMS, TOY_LENGTH))

rng_spike = np.random.default_rng(7)
rng_shift = np.random.default_rng(8)
rng_drift = np.random.default_rng(9)
rng_freq = np.random.default_rng(10)
rng_clean = np.random.default_rng(11)

spike_signal = np.sin(t) + rng_spike.normal(0, CORPUS_NOISE, size=len(t))
spike_index = np.abs(t - (np.pi / 2)).argmin()
spike_signal[spike_index] += 3.0

level_shift_signal = np.sin(t) + rng_shift.normal(0, CORPUS_NOISE, size=len(t))
shift_start = np.where(t > np.pi)[0][0]
level_shift_signal[shift_start:] += 2.0

drift_signal = t + np.sin(t) + rng_drift.normal(0, CORPUS_NOISE, size=len(t))
freq_signal = np.sin(2 * t) + rng_freq.normal(0, CORPUS_NOISE, size=len(t))

multi_spike_signal = np.sin(t) + rng_spike.normal(0, CORPUS_NOISE, size=len(t))
multi_spike_index = np.abs(t - (3 * np.pi / 2)).argmin()
multi_spike_index2 = np.abs(t - (5 * np.pi / 2)).argmin()
for index in (spike_index, multi_spike_index, multi_spike_index2):
    multi_spike_signal[index] += 3.0

no_anomaly_signal = np.sin(t) + rng_clean.normal(0, CORPUS_NOISE, size=len(t))

rng_sq = {name: np.random.default_rng(seed) for name, seed in [
    ('spike', 21), ('level_shift', 22), ('upward_drift', 23), ('doubled_frequency', 24),
    ('no_anomaly', 25), ('multi_spike', 26), ('half_and_half', 27)]}

sq_spike = square_base + rng_sq['spike'].normal(0, CORPUS_NOISE, size=len(t))
sq_spike[spike_index] += 3.0

sq_level_shift = square_base + rng_sq['level_shift'].normal(0, CORPUS_NOISE, size=len(t))
sq_level_shift[shift_start:] += 2.0

sq_upward_drift = t + square_base + rng_sq['upward_drift'].normal(0, CORPUS_NOISE, size=len(t))
sq_doubled_frequency = square_wave(2 * t) + rng_sq['doubled_frequency'].normal(
    0, CORPUS_NOISE, size=len(t))
sq_no_anomaly = square_base + rng_sq['no_anomaly'].normal(0, CORPUS_NOISE, size=len(t))

sq_multi_spike = square_base + rng_sq['multi_spike'].normal(0, CORPUS_NOISE, size=len(t))
for index in (spike_index, multi_spike_index, multi_spike_index2):
    sq_multi_spike[index] += 3.0

half_and_half = (np.where(t < 2 * np.pi, np.sin(t), square_base)
                 + rng_sq['half_and_half'].normal(0, CORPUS_NOISE, size=len(t)))

NOISY_VARIANCE = 1.0
NOISY_SD = np.sqrt(NOISY_VARIANCE)

sine_noisy = np.sin(t) + np.random.default_rng(12).normal(0, NOISY_SD, size=len(t))
square_noisy = square_base + np.random.default_rng(28).normal(0, NOISY_SD, size=len(t))

ANOMALY_ORDER = ['spike', 'level_shift', 'upward_drift', 'doubled_frequency',
                 'no_anomaly', 'multi_spike']

toy_tests = (
    [('sine_' + name, values) for name, values in zip(ANOMALY_ORDER, [
        spike_signal, level_shift_signal, drift_signal, freq_signal,
        no_anomaly_signal, multi_spike_signal])]
    + [('square_' + name, values) for name, values in zip(ANOMALY_ORDER, [
        sq_spike, sq_level_shift, sq_upward_drift, sq_doubled_frequency,
        sq_no_anomaly, sq_multi_spike])]
    + [('half_and_half', half_and_half),
       ('sine_noisy', sine_noisy),
       ('square_noisy', square_noisy)]
)



### 1.1 Building the merged corpus and scoring it
Constructs the corpus of streams

In [ ]:
%%capture
%%time
%%capture
def toy_canonical(streams):
    return to_canonical(
        [(name, pd.DataFrame({'time': t, 'output': np.asarray(values, dtype=float)}))
         for name, values in streams], time_col='time')

mm_rng = np.random.default_rng(0)
sine_subset = toy_corpus[mm_rng.choice(len(toy_corpus), size=TOY_CORPUS_N, replace=False)]
square_subset = square_corpus[mm_rng.choice(len(square_corpus), size=TOY_CORPUS_N,
                                            replace=False)]

TOY_CORPORA = {
    'merged': [('sine_{0:05d}'.format(i), v) for i, v in enumerate(sine_subset)]
              + [('square_{0:05d}'.format(i), v) for i, v in enumerate(square_subset)],
    'sine only': [('sine_{0:05d}'.format(i), v) for i, v in enumerate(sine_subset)],
    'square only': [('square_{0:05d}'.format(i), v) for i, v in enumerate(square_subset)],
}

toy_test_canonical = toy_canonical(toy_tests)

toy_thresholds = {}
toy_corpora_frames = {}
toy_covariances = {}


def run_toy(reference, label):
    corpus = compute_corpus(toy_canonical(reference), trunc=TOY_TRUNC,
                            granularity=TOY_GRANULARITY)

    covariance_path = SCRATCH / 'toy_{0}_covariance.npz'.format(label.replace(' ', '_'))
    _, info = create_covariance(corpus, output_path=covariance_path,
                                variance_keep=TOY_VARIANCE_KEEP, form='inv_sqrt',
                                factored=CONFIG['metric']['factored'])
    covariance = load_whitening(covariance_path)

    thresholds, _ = crossvalidated_thresholds(
        corpus, covariance, k=FOLDS, statistic=STATISTIC, band=BAND,
        neighbours=TOY_NEIGHBOURS, windowed=False, show_progress=True)

    toy_thresholds[label] = thresholds
    toy_corpora_frames[label] = corpus
    toy_covariances[label] = covariance
    index_path = SCRATCH / 'toy_{0}_index.faiss'.format(label.replace(' ', '_'))
    meta_path = SCRATCH / 'toy_{0}_meta.npz'.format(label.replace(' ', '_'))
    build_index({'corpus': corpus}, covariance, output_index=index_path,
                output_meta=meta_path, band=BAND)
    index = load_index(index_path, meta_path)

    scored = score_streams(toy_test_canonical, index, covariance, thresholds,
                           trunc=TOY_TRUNC, neighbours=TOY_NEIGHBOURS,
                           sig_tol=TOY_SIG_TOL)
    toy_diagnostics.append({
        'corpus': label, 'intervals': len(corpus), 'rank': info['rank'],
        'streams_flagged': int(scored.anomalous.map(len).gt(0).sum())})
    return scored


toy_diagnostics = []
toy_results = {label: run_toy(reference, label)
               for label, reference in TOY_CORPORA.items()}
toy_diagnostics = pd.DataFrame(toy_diagnostics)


### 1.2 What each corpus flags

In [ ]:
%%capture
rows = []
for label, scored in toy_results.items():
    flags = dict(zip(scored.stream, scored.anomalous))
    for name, _ in toy_tests:
        intervals = flags.get(name, [])
        covered = sum(hi - lo + 1 for lo, hi in intervals)
        rows.append({'stream': name, 'corpus': label,
                     'intervals': len(intervals),
                     'flagged_fraction': covered / TOY_LENGTH})

toy_table = (pd.DataFrame(rows)
             .pivot(index='stream', columns='corpus', values='flagged_fraction')
             .reindex([name for name, _ in toy_tests])
             [list(TOY_CORPORA)])
clean = ['sine_no_anomaly', 'square_no_anomaly']


### 1.3 Where the flags land

In [ ]:
merged_flags = dict(zip(toy_results['merged'].stream, toy_results['merged'].anomalous))

fig, axes = plt.subplots(4, 4, figsize=(20, 10), sharex=True)
for ax, (label, values) in zip(axes.ravel(), toy_tests):
    ax.plot(t, values, lw=1.0, color='#1f77b4')
    for lo, hi in merged_flags.get(label, []):
        ax.axvspan(t[int(lo)], t[int(min(hi, len(t) - 1))],
                   color='darkorange', alpha=0.30, lw=0)
    covered = sum(hi - lo + 1 for lo, hi in merged_flags.get(label, [])) / TOY_LENGTH
    ax.set_title('{0} - {1:.0%} flagged'.format(label, covered), fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
for ax in axes.ravel()[len(toy_tests):]:
    ax.axis('off')
fig.suptitle('Merged corpus (sine + square): what the widest-first search flags, and where',
             y=1.0)
plt.tight_layout()
FIGURES = PROJECT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

np.savez(FIGURES / 'toy_merged_flags.npz', t=t,
         names=np.array([name for name, _ in toy_tests]),
         signals=np.array([np.asarray(v, dtype=float) for _, v in toy_tests]),
         flags=np.array(json.dumps(
             {name: [[int(lo), int(hi)] for lo, hi in merged_flags.get(name, [])]
              for name, _ in toy_tests})))

try:
    fig.savefig(FIGURES / 'toy_merged_flags.png', dpi=200, bbox_inches='tight',
                facecolor='white')
except OSError:
    spare = FIGURES / 'toy_merged_flags.new.png'
    fig.savefig(spare, dpi=200, bbox_inches='tight', facecolor='white')
    print('toy_merged_flags.png was locked; wrote {0} instead'.format(spare.name))

plt.show()

### 1.4 Where to set the threshold

Determining an optimal operating threshold is key to efficacy, this section explores a supervised and unsupervised methodologies for this problem.

In [ ]:
%%time
from anomalies_scale.AUC import POINT_SCORE_COLUMN, ranking_metrics, score_points

TOY_WINDOW = 2 ** TOY_GRANULARITY
VALIDATION_N = 200
VALIDATION_FACTORS = (0.5, 1.0, 1.5)
VALIDATION_PERCENTILE = 95
THRESHOLD_CORPUS = 'merged'


def toy_point_truth(name):
    """Which points of a test stream are anomalous, by construction rather than by detection."""
    mask = np.zeros(TOY_LENGTH, dtype=bool)
    if name.endswith('multi_spike'):
        mask[[spike_index, multi_spike_index, multi_spike_index2]] = True
    elif name.endswith('spike'):
        mask[spike_index] = True
    elif name.endswith('level_shift'):
        mask[shift_start:] = True
    elif name.endswith(('upward_drift', 'doubled_frequency')):
        mask[:] = True
    return mask


# ignoring 'half and half' as it is ambiguous
THRESHOLD_TESTS = [(name, values) for name, values in toy_tests if name != 'half_and_half']

validation_rng = np.random.default_rng(2024)
per_cell = VALIDATION_N // (2 * len(VALIDATION_FACTORS))
toy_validation, validation_factor_of = [], {}
for factor in VALIDATION_FACTORS:
    noise = CORPUS_NOISE * factor
    for mode, base in (('sine', np.sin(t)), ('square', square_base)):
        for i in range(per_cell):
            name = 'val_{0}_x{1:g}_{2:04d}'.format(mode, factor, i)
            toy_validation.append(
                (name, base + validation_rng.normal(0, noise, size=TOY_LENGTH)))
            validation_factor_of[name] = factor

stem = THRESHOLD_CORPUS.replace(' ', '_')
threshold_covariance = load_whitening(SCRATCH / 'toy_{0}_covariance.npz'.format(stem))
threshold_index = load_index(SCRATCH / 'toy_{0}_index.faiss'.format(stem),
                             SCRATCH / 'toy_{0}_meta.npz'.format(stem))
calibrated = toy_thresholds[THRESHOLD_CORPUS]


def point_scores(streams):
    scored = score_points(toy_canonical(streams), threshold_index, threshold_covariance,
                          calibrated, window=TOY_WINDOW, trunc=TOY_TRUNC,
                          neighbours=TOY_NEIGHBOURS)
    return dict(zip(scored.stream, scored[POINT_SCORE_COLUMN]))


test_scores = point_scores(THRESHOLD_TESTS)
validation_scores = point_scores(toy_validation)

toy_scores = np.concatenate([np.asarray(test_scores[name], dtype=float)
                             for name, _ in THRESHOLD_TESTS])
toy_truth = np.concatenate([toy_point_truth(name) for name, _ in THRESHOLD_TESTS])
validation_pool = np.concatenate([np.asarray(v, dtype=float)
                                  for v in validation_scores.values()])

ranking = ranking_metrics(toy_scores, toy_truth)


def f1_optimal(scores, truth):
    from sklearn.metrics import precision_recall_curve

    precision, recall, cuts = precision_recall_curve(truth, scores)
    harmonic = 2 * precision[:-1] * recall[:-1] / np.maximum(
        precision[:-1] + recall[:-1], 1e-12)
    return float(cuts[int(np.nanargmax(harmonic))])


def quality(cut):
    from sklearn.metrics import precision_recall_fscore_support

    flagged = toy_scores >= cut
    precision, recall, f1, _ = precision_recall_fscore_support(
        toy_truth, flagged, average='binary', zero_division=0)
    return {'cut': cut, 'flagged': float(flagged.mean()),
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
            'validation_flagged': float((validation_pool >= cut).mean())}


TOY_THRESHOLDS = {
    # `score_points` divides by the calibrated threshold, so the calibrated cut is 1.0 here.
    'calibrated (cross-validated)': 1.0,
    'validation (p{0}, {1} x corpus noise)'.format(
        VALIDATION_PERCENTILE, '/'.join('{0:g}'.format(f) for f in VALIDATION_FACTORS)):
        float(np.percentile(validation_pool, VALIDATION_PERCENTILE)),
    'oracle (F1-optimal)': f1_optimal(toy_scores, toy_truth),
}

threshold_table = pd.DataFrame([dict(threshold=name, **quality(cut))
                                for name, cut in TOY_THRESHOLDS.items()])
display(summarise(corpus=THRESHOLD_CORPUS, window=TOY_WINDOW,
                  test_points=int(toy_truth.size),
                  anomalous_points=int(toy_truth.sum()),
                  anomaly_rate=round(float(toy_truth.mean()), 4),
                  validation_streams=len(toy_validation),
                  roc_auc=round(ranking.get('roc_auc', float('nan')), 4),
                  pr_auc=round(ranking.get('pr_auc', float('nan')), 4),
                  pr_auc_lift=round(ranking.get('pr_auc_lift', float('nan')), 2)))
display(threshold_table.style.format({
    'cut': '{:.3f}', 'flagged': '{:.1%}', 'precision': '{:.3f}', 'recall': '{:.3f}',
    'f1': '{:.3f}', 'validation_flagged': '{:.1%}'}).hide(axis='index'))

by_factor = pd.DataFrame([
    {'noise factor': factor,
     'streams': sum(1 for f in validation_factor_of.values() if f == factor),
     **{name: float((np.concatenate(
         [np.asarray(validation_scores[s], dtype=float)
          for s, f in validation_factor_of.items() if f == factor]) >= cut).mean())
        for name, cut in TOY_THRESHOLDS.items()}}
    for factor in VALIDATION_FACTORS])
display(by_factor.style.format(
    {name: '{:.1%}' for name in TOY_THRESHOLDS}).hide(axis='index')
    .set_caption('fraction of normal validation points flagged, by noise level'))

fig, ax = plt.subplots(figsize=(11, 4.6))
bins = np.linspace(0, np.percentile(toy_scores, 99.5), 60)
ax.hist(toy_scores[~toy_truth], bins=bins, color='#1d4ed8', alpha=0.55, label='normal points')
ax.hist(toy_scores[toy_truth], bins=bins, color='#b91c1c', alpha=0.65, label='anomalous points')
ax.hist(validation_pool, bins=bins, color='#15803d', alpha=0.35, label='validation (normal)')
for (name, cut), style in zip(TOY_THRESHOLDS.items(), ('-', '--', ':')):
    ax.axvline(cut, color='#0f172a', ls=style, lw=1.4, label='{0} = {1:.2f}'.format(name, cut))
ax.set_yscale('log')
ax.set_xlabel('score (distance / calibrated threshold)')
ax.set_ylabel('points')
ax.legend(fontsize=8)
ax.set_title('Toy: score distribution and three thresholds', fontsize=12)
plt.tight_layout()

FIGURES = PROJECT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
try:
    fig.savefig(FIGURES / 'toy_thresholds.png', dpi=200, bbox_inches='tight',
                facecolor='white')
except OSError:
    print('toy_thresholds.png was locked; not rewritten')
plt.show()


## Comparing the supervised and unsupervised thresholds
Both unsupervised thresholds are a quantile of a distance distribution, so the choice of
quantile is the only free parameter either has. The calibrated one quantiles corpus distances
with each fold held out; the validation one quantiles fresh normal streams at twice the corpus
noise. The oracle marks what the best possible cut achieves.

In [ ]:
%%time
from anomalies_scale.crossvalidated_thresholding import fold_distances, stream_folds
from anomalies_scale.covariance_creation import signature_matrix

SWEEP_PERCENTILES = [50, 75, 90, 95, 97.5, 99, 99.5, 99.9, 100]

sweep_corpus = toy_corpora_frames[THRESHOLD_CORPUS]
sweep_covariance = toy_covariances[THRESHOLD_CORPUS]

SWEEP_CACHE = SCRATCH / 'toy_{0}_folddistances_k{1}_n{2}_b{3}.npz'.format(
    THRESHOLD_CORPUS.replace(' ', '_'), FOLDS, TOY_NEIGHBOURS, BAND)

signatures, _ = signature_matrix(sweep_corpus)
whitened = np.ascontiguousarray(sweep_covariance.apply(signatures), dtype='float32')
sweep_depths = sweep_corpus['depth'].to_numpy(dtype=int)
sweep_streams = sweep_corpus['stream'].to_numpy()
if SWEEP_CACHE.exists():
    calibration_distances = np.load(SWEEP_CACHE)['distances']
    print('loaded cached fold distances from {0}'.format(SWEEP_CACHE.name))
else:
    calibration_distances = fold_distances(
        whitened, sweep_depths, sweep_streams,
        stream_folds(sweep_streams, FOLDS, CONFIG['calibrate']['random_state']),
        band=BAND, neighbours=TOY_NEIGHBOURS)
    np.savez_compressed(SWEEP_CACHE, distances=calibration_distances)
    print('wrote {0}'.format(SWEEP_CACHE.name))

# The window 1.4 scores at resolves to one depth, so both thresholds are scalars here.
sweep_depth = threshold_index.depth_for_width(TOY_WINDOW / float(TOY_LENGTH - 1))
calibration_at_depth = calibration_distances[sweep_depths == sweep_depth]


def raw_point_scores(streams):
    """Distances rather than ratios: `threshold=1.0` leaves `score_points` undivided."""
    scored = score_points(toy_canonical(streams), threshold_index, sweep_covariance, 1.0,
                          window=TOY_WINDOW, trunc=TOY_TRUNC, neighbours=TOY_NEIGHBOURS)
    return dict(zip(scored.stream, scored[POINT_SCORE_COLUMN]))


raw_test = raw_point_scores(THRESHOLD_TESTS)
raw_validation = raw_point_scores(toy_validation)
raw_scores = np.concatenate([np.asarray(raw_test[name], dtype=float)
                             for name, _ in THRESHOLD_TESTS])
raw_validation_pool = np.concatenate([np.asarray(v, dtype=float)
                                      for v in raw_validation.values()])

oracle_cut = f1_optimal(raw_scores, raw_truth := toy_truth)


def sweep_quality(cut):
    from sklearn.metrics import precision_recall_fscore_support

    flagged = raw_scores >= cut
    precision, recall, f1, _ = precision_recall_fscore_support(
        toy_truth, flagged, average='binary', zero_division=0)
    return {'cut': float(cut), 'flagged': float(flagged.mean()),
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
            'validation_flagged': float((raw_validation_pool >= cut).mean())}


sweep_rows = []
for percentile in SWEEP_PERCENTILES:
    for source, values in (('calibrated', calibration_at_depth),
                           ('validation', raw_validation_pool)):
        sweep_rows.append(dict(source=source, percentile=percentile,
                               **sweep_quality(np.percentile(values, percentile))))

toy_sweep = pd.DataFrame(sweep_rows)
oracle = sweep_quality(oracle_cut)

RESULTS.mkdir(parents=True, exist_ok=True)
toy_sweep.to_csv(RESULTS / 'toy_threshold_sweep.csv', index=False)
pd.DataFrame([dict(source='oracle', percentile=float('nan'), **oracle)]).to_csv(
    RESULTS / 'toy_threshold_oracle.csv', index=False)

display(toy_sweep.pivot(index='percentile', columns='source',
                        values=['cut', 'f1', 'validation_flagged'])
        .style.format('{:.3f}').set_caption(
            'oracle cut {0:.2f}, F1 {1:.3f}, false alarms {2:.1%}'.format(
                oracle['cut'], oracle['f1'], oracle['validation_flagged'])))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
COLOURS = {'calibrated': '#1d4ed8', 'validation': '#15803d'}
x = np.arange(len(SWEEP_PERCENTILES))

for ax, column, label, logscale in (
        (axes[0], 'cut', 'threshold (distance)', True),
        (axes[1], 'f1', 'F1 on the test streams', False)):
    for source, colour in COLOURS.items():
        block = toy_sweep[toy_sweep.source == source].set_index('percentile')
        ax.plot(x, [block.loc[p, column] for p in SWEEP_PERCENTILES], marker='o', ms=4,
                color=colour, label=source)
    ax.axhline(oracle[column], color='#b91c1c', ls='--', lw=1.3, label='oracle')
    if logscale:
        ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels([('{0:g}'.format(p)) for p in SWEEP_PERCENTILES], fontsize=8)
    ax.set_xlabel('percentile')
    ax.set_ylabel(label)
    ax.grid(color='#e2e8f0')
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(fontsize=9)

fig.suptitle('Toy: percentile sweep for both thresholds', fontsize=12)
plt.tight_layout()
try:
    fig.savefig(FIGURES / 'toy_threshold_sweep.png', dpi=200, bbox_inches='tight',
                facecolor='white')
except OSError:
    print('toy_threshold_sweep.png was locked; not rewritten')
plt.show()


## 2. C-MAPSS

In [ ]:
%%capture
import time

from anomalies_scale.throughput import peak_memory_bytes

STAGE_SECONDS = {}

started = time.perf_counter()
from anomalies_scale.canonical_streams import to_canonical
from anomalies_scale.cmapss_labels import labels_for_run_to_failure
from anomalies_scale.windowing import window_streams, reassemble
from anomalies_scale.signature_computer import compute_corpus
from anomalies_scale.covariance_creation import create_covariance, load_whitening
from anomalies_scale.crossvalidated_thresholding import crossvalidated_thresholds
from anomalies_scale.index_creation import build_index, load_index
from anomalies_scale.stream_scoring import score_streams
from anomalies_scale.stream_evaluation import (
    evaluate_predictions, intervals_to_mask, segments)

SUBSET = 'FD001'

WINDOW_SIZE = CONFIG['window']['size']
GRANULARITY = CONFIG['signature']['granularity']
TRUNC = CONFIG['signature']['trunc']
NEIGHBOURS = CONFIG['detect']['neighbours']
STATISTIC = CONFIG['calibrate']['statistic']
FOLDS = CONFIG['calibrate']['folds']
BAND = CONFIG['detect']['band']

VARIANCE_KEEP = 0.99

SCRATCH = PROJECT / 'data' / 'notebook'
SCRATCH.mkdir(parents=True, exist_ok=True)

corpus_half, eval_half = CMAPSS_DATA[SUBSET]

def canonicalise(frame):
    frame = frame.sort_values(['unit', 'cycle'])
    return to_canonical([
        ('{0}_unit{1:03d}'.format(SUBSET, int(unit)),
         engine.rename(columns={'cycle': 'time'})[['time'] + CHANNELS])
        for unit, engine in frame.groupby('unit', sort=True)
    ], time_col='time')

corpus_canonical = canonicalise(corpus_half)
eval_canonical = canonicalise(eval_half)

eval_labels = labels_for_run_to_failure(
    eval_canonical, horizon=HEALTH_HORIZON, show_progress=True)

corpus_windows = window_streams(corpus_canonical, WINDOW_SIZE,
                                min_points=2 ** GRANULARITY + 1, show_progress=True)
eval_windows = window_streams(eval_canonical, WINDOW_SIZE, min_points=2, show_progress=True)

window_counts = pd.DataFrame([
    {'side': 'corpus', 'windows': len(corpus_windows),
     'engines': corpus_windows.source.nunique()},
    {'side': 'evaluation', 'windows': len(eval_windows),
     'engines': eval_windows.source.nunique()}])
display(window_counts)
STAGE_SECONDS['streams_windows'] = time.perf_counter() - started

started = time.perf_counter()
corpus_signatures = compute_corpus(
    corpus_windows, trunc=TRUNC, granularity=GRANULARITY, show_progress=True)
STAGE_SECONDS['corpus'] = time.perf_counter() - started

started = time.perf_counter()
covariance_path = SCRATCH / '{0}_covariance.npz'.format(SUBSET)
_, metric = create_covariance(
    corpus_signatures, output_path=covariance_path, variance_keep=VARIANCE_KEEP,
    form='inv_sqrt', factored=CONFIG['metric']['factored'], show_progress=True)
covariance = load_whitening(covariance_path)

display(summarise(dimension=metric['dimension'], rank=metric['rank'],
                  variance_retained='{0:.4%}'.format(metric['variance_retained']),
                  condition_number='{0:.1e}'.format(metric['condition_number']),
                  stored='factored' if covariance.factored else 'dense',
                  latent_width=covariance.latent,
                  signature_width=covariance.dimension))
STAGE_SECONDS['covariance'] = time.perf_counter() - started

started = time.perf_counter()
thresholds, fold_table = crossvalidated_thresholds(
    corpus_signatures, covariance, k=FOLDS, statistic=STATISTIC, band=BAND,
    neighbours=NEIGHBOURS, windowed=True, show_progress=True)

display(pd.DataFrame({'threshold': pd.Series(thresholds)}).rename_axis('depth')
        .style.format('{:,.1f}')
        .set_caption('per-depth thresholds at {0}, k={1}, {2} neighbour(s)'.format(
            STATISTIC, FOLDS, NEIGHBOURS)))
STAGE_SECONDS['calibrate'] = time.perf_counter() - started

started = time.perf_counter()
index_path = SCRATCH / '{0}_index.faiss'.format(SUBSET)
meta_path = SCRATCH / '{0}_index_meta.npz'.format(SUBSET)

diagnostics = build_index({'corpus': corpus_signatures}, covariance,
                          output_index=index_path, output_meta=meta_path,
                          band=BAND, show_progress=True)
index = load_index(index_path, meta_path, index_type=CONFIG['detect']['index']['type'])

scored_windows = score_streams(
    eval_windows, index, covariance, thresholds, trunc=TRUNC, span=WINDOW_SIZE,
    neighbours=NEIGHBOURS, show_progress=True)

scored = reassemble(scored_windows)
display(summarise(scored_windows=len(scored_windows), engines=len(scored),
                  flagged_intervals=int(scored.anomalous.map(len).sum())))
STAGE_SECONDS['index_and_score'] = time.perf_counter() - started

started = time.perf_counter()
summary = evaluate_predictions(scored, truth=eval_labels, show_progress=True)
STAGE_SECONDS['evaluate'] = time.perf_counter() - started

BUILD_STAGES = ('streams_windows', 'corpus', 'covariance', 'calibrate')
SERVE_STAGES = ('index_and_score',)

counts = {
    'corpus_windows': len(corpus_windows),
    'corpus_intervals': len(corpus_signatures),
    'signature_terms': int(metric['dimension']),
    'retained_rank': int(metric['rank']),
    'test_windows': len(eval_windows),
    'points_scored': int(summary['n_points']),
}

build_seconds = sum(STAGE_SECONDS[s] for s in BUILD_STAGES if s in STAGE_SECONDS)
serve_seconds = sum(STAGE_SECONDS[s] for s in SERVE_STAGES if s in STAGE_SECONDS)
total = sum(STAGE_SECONDS.values())

stage_table = pd.DataFrame({'seconds': pd.Series(STAGE_SECONDS)}).rename_axis('stage')
stage_table['share'] = stage_table.seconds / max(total, 1e-9)
stage_table.loc['total'] = [total, 1.0]

count_table = pd.DataFrame({'count': pd.Series(
    {'corpus windows': counts['corpus_windows'],
     'corpus intervals': counts['corpus_intervals'],
     'signature terms': counts['signature_terms'],
     'retained rank': counts['retained_rank'],
     'test windows': counts['test_windows'],
     'points scored': counts['points_scored']})})

throughput_table = summarise(
    build_seconds=round(build_seconds, 2), serve_seconds=round(serve_seconds, 2),
    build_per_serve_second=round(build_seconds / max(serve_seconds, 1e-9), 1),
    corpus_intervals_per_second=round(
        counts['corpus_intervals'] / max(STAGE_SECONDS.get('corpus', 0), 1e-9)),
    points_per_second=round(counts['points_scored'] / max(serve_seconds, 1e-9)),
    peak_rss_mb=round(peak_memory_bytes() / 1e6))

display(stage_table.style.format({'seconds': '{:.2f}', 'share': '{:.1%}'}),
        count_table.style.format('{:,}'), throughput_table)

eval_sorted = eval_half.sort_values(['unit', 'cycle'])
order = list(scored.stream)
truth_by_stream = dict(zip(eval_labels.stream, eval_labels.label))
lengths = {name: len(truth_by_stream[name]) for name in order}

mask = np.concatenate([intervals_to_mask(intervals, lengths[name])
                       for name, intervals in zip(order, scored.anomalous)])
truth = np.concatenate([np.asarray(truth_by_stream[name], dtype=bool) for name in order])
eval_rul = np.concatenate([
    eval_sorted[eval_sorted.unit == int(name.split('unit')[1])].rul.to_numpy()
    for name in order])

varying = [c for c in CHANNELS if corpus_half[c].std() > 0]
SHOW = list(corpus_half[varying].corrwith(corpus_half.rul).abs()
            .sort_values(ascending=False).index[:5])
lifetime = eval_sorted.groupby('unit').size()
EVAL_UNIT = int((lifetime - lifetime.median()).abs().idxmin())

offsets = np.cumsum([0] + [lengths[name] for name in order])


### 2.1 Eight engines across the lifetime range

In [ ]:
sample_units = lifetime.sort_values().iloc[
    np.linspace(0, len(lifetime) - 1, 8).round().astype(int)].index

fig, axes = plt.subplots(4, 2, figsize=(15, 10), sharey=True)
for ax, unit in zip(axes.ravel(), sample_units):
    name = '{0}_unit{1:03d}'.format(SUBSET, int(unit))
    at = order.index(name)
    rows = np.arange(offsets[at], offsets[at + 1])
    sub = eval_sorted[eval_sorted.unit == unit]
    ax.plot(sub.cycle, sub[SHOW[0]], lw=0.6, color='#1f77b4')

    low, high = sub[SHOW[0]].min(), sub[SHOW[0]].max()
    pad = (high - low) * 0.12
    for series, colour, offset in ((truth[rows], 'crimson', 0),
                                   (mask[rows], 'darkorange', 1)):
        for lo, hi in segments(series):
            ax.fill_between([sub.cycle.iloc[lo], sub.cycle.iloc[hi - 1]],
                            low - pad * (offset + 1), low - pad * offset,
                            color=colour, lw=0)

    ax.set_title('unit {0} - {1} cycles'.format(unit, len(rows)), fontsize=9)
    ax.margins(x=0)
    ax.set_ylim(low - pad * 2.4, high)

for ax in axes[-1]:
    ax.set_xlabel('cycle')
fig.suptitle('{0} across eight engines. Upper bar = labelled degraded (RUL <= {1}), '
             'lower bar = flagged by the pipeline detector'.format(SHOW[0], HEALTH_HORIZON),
             y=1.0)
plt.tight_layout()
plt.show()

### 2.2 The health curves

### A caveat first: C-MAPSS is a prognostics benchmark, not an anomaly-detection one

C-MAPSS (Saxena et al., PHM 2008) was built for remaining-useful-life regress, not an anomaly detection dataset.The run-to-failure half has none, and the truncated half
comes with a single integer per engine - the cycles it would have survived past the end of its
record. Nothing says which cycles are anomalous.

So the labels everything above is scored against are slightly arbitrary. The convention
used here is the piecewise-linear RUL target standard in the prognostics literature: sensor
response to damage is not considered observable until roughly the last 125-130 cycles, and a
cycle is called degraded when its remaining life is at or below `HEALTH_HORIZON`. Three
consequences follow, and they are worth stating plainly rather than leaving in a footnote:

### Why this section is the part that does not depend on any of that

The curve plots the detector's score against remaining useful life, and **remaining life is
something the data states exactly**: in the run-to-failure half the record ends at failure, so
the remaining life at cycle *i* of an *L*-cycle engine is `L - 1 - i`, read off the record
rather than declared. Both axes are therefore measured quantities. The labelled bands are
shaded for orientation only, and nothing in the curve is computed from them.

That makes this the one result here that a different labelling convention cannot move. If the
score rises smoothly as remaining life falls, it does so whatever anyone decides to call
degraded; the convention only decides where the horizontal line gets drawn. And the shape is
the interesting part - a score that is flat until the last few cycles and then jumps would mean
the method detects failure, while one that climbs steadily through the ambiguous band means it
anticipates it.

C-MAPSS degradation is gradual and largely monotone drift in sensor
levels, which is a favourable case for any method keyed to level and spread - and therefore a
poor place to demonstrate what signatures uniquely add, which is sensitivity to ordering,
lead-lag and multichannel interaction. Doing well here is necessary but not sufficient, and it
is precisely why the comparison against ordinary summary statistics matters.

In [ ]:
%%capture
%%time
from anomalies_scale.AUC import (
    POINT_SCORE_COLUMN, ranking_metrics, score_points)

SLIDE = CONFIG['evaluate']['point_scores']['window']
SLIDE_STRIDE = CONFIG['evaluate']['point_scores']['stride']

def health_curve(frame, column='ratio'):
    return frame.groupby('end_rul').agg(
        mean_score=(column, 'mean'),
        median_score=(column, 'median'),
        q25=(column, lambda v: v.quantile(0.25)),
        q75=(column, lambda v: v.quantile(0.75)),
        n_engines=('unit', 'nunique')).reset_index()

import gc

MIN_ENGINES = 20

def canonicalise_subset(frame, subset):
    frame = frame.sort_values(['unit', 'cycle'])
    return to_canonical([
        ('{0}_unit{1:03d}'.format(subset, int(unit)),
         engine.rename(columns={'cycle': 'time'})[['time'] + CHANNELS])
        for unit, engine in frame.groupby('unit', sort=True)
    ], time_col='time')

def build_detector(subset):
    corpus_half, eval_half = CMAPSS_DATA[subset]
    corpus_canonical = canonicalise_subset(corpus_half, subset)
    eval_canonical = canonicalise_subset(eval_half, subset)
    labels = labels_for_run_to_failure(eval_canonical, horizon=HEALTH_HORIZON)

    corpus_windows = window_streams(corpus_canonical, WINDOW_SIZE,
                                    min_points=2 ** GRANULARITY + 1)
    eval_windows = window_streams(eval_canonical, WINDOW_SIZE, min_points=2)
    corpus = compute_corpus(corpus_windows, trunc=TRUNC, granularity=GRANULARITY)

    covariance_path = SCRATCH / '{0}_covariance.npz'.format(subset)
    _, metric = create_covariance(
        corpus, output_path=covariance_path, variance_keep=VARIANCE_KEEP,
        form='inv_sqrt', factored=CONFIG['metric']['factored'])
    covariance = load_whitening(covariance_path)

    thresholds, _ = crossvalidated_thresholds(
        corpus, covariance, k=FOLDS, statistic=STATISTIC, band=BAND,
        neighbours=NEIGHBOURS, windowed=True)

    index_path = SCRATCH / '{0}_index.faiss'.format(subset)
    meta_path = SCRATCH / '{0}_index_meta.npz'.format(subset)
    build_index({'corpus': corpus}, covariance, output_index=index_path,
                output_meta=meta_path, band=BAND)
    index = load_index(index_path, meta_path, index_type=CONFIG['detect']['index']['type'])

    scored = reassemble(score_streams(
        eval_windows, index, covariance, thresholds, trunc=TRUNC, span=WINDOW_SIZE,
        neighbours=NEIGHBOURS))
    summary = evaluate_predictions(scored, truth=labels)

    points = score_points(
        eval_canonical, index, covariance, thresholds, window=SLIDE, stride=SLIDE_STRIDE,
        span=WINDOW_SIZE, trunc=TRUNC, neighbours=NEIGHBOURS)

    ratio_by_stream = dict(zip(points['stream'], points[POINT_SCORE_COLUMN]))
    truth_by_stream = dict(zip(labels.stream, labels.label))
    order = list(points['stream'])
    ordered_eval = eval_half.sort_values(['unit', 'cycle'])

    units = [int(name.split('unit')[1]) for name in order]
    slide = pd.DataFrame({
        'unit': np.concatenate([np.full(len(truth_by_stream[name]), unit)
                                for name, unit in zip(order, units)]),
        'end_rul': np.concatenate([ordered_eval[ordered_eval.unit == unit].rul.to_numpy()
                                   for unit in units]),
        'ratio': np.concatenate([np.asarray(ratio_by_stream[name], dtype=float)
                                 for name in order]),
        'degraded': np.concatenate([np.asarray(truth_by_stream[name], dtype=bool)
                                    for name in order]),
    })

    curve = health_curve(slide)
    counts = {'corpus_windows': len(corpus_windows), 'corpus_intervals': len(corpus),
              'rank': metric['rank'], 'engines': len(labels)}

    del corpus, corpus_windows, eval_windows, corpus_canonical, eval_canonical, index
    gc.collect()
    return {'curve': curve, 'slide': slide, 'summary': summary, 'counts': counts,
            'thresholds': thresholds}


ZSCORE_WINDOW = 15


def rolling_zscore(subset, window=ZSCORE_WINDOW):
    """Per-cycle deviation from the engine's own trailing window, over every channel."""
    _, evaluation = CMAPSS_DATA[subset]
    frame = evaluation.sort_values(['unit', 'cycle'])

    pieces = []
    for unit, engine in frame.groupby('unit', sort=True):
        values = engine[CHANNELS]
        centre = values.rolling(window, min_periods=window).mean()
        spread = values.rolling(window, min_periods=window).std(ddof=0)
        deviations = ((values - centre) / spread.where(spread > 0)).abs()
        pieces.append(pd.DataFrame({
            'unit': int(unit),
            'end_rul': engine.rul.to_numpy(),
            'zmax': deviations.max(axis=1).to_numpy()}))

    out = pd.concat(pieces, ignore_index=True)
    return out.dropna(subset=['zmax']).reset_index(drop=True)


STOMP_WINDOW = ZSCORE_WINDOW


def _rolling_stats(series, m):
    """Mean and standard deviation of every length-m window, by cumulative sums."""
    total = np.concatenate([[0.0], np.cumsum(series)])
    square = np.concatenate([[0.0], np.cumsum(series ** 2)])
    mean = (total[m:] - total[:-m]) / m
    variance = np.maximum((square[m:] - square[:-m]) / m - mean ** 2, 0.0)
    return mean, np.sqrt(variance)


def stomp(series, m, exclusion=None):
    """Matrix profile: distance from each subsequence to its nearest neighbour elsewhere."""
    series = np.asarray(series, dtype=float)
    n = len(series) - m + 1
    if n < 2:
        return np.zeros(max(n, 0))
    exclusion = int(np.ceil(m / 4)) if exclusion is None else int(exclusion)

    mean, sigma = _rolling_stats(series, m)
    safe = np.where(sigma > 1e-12, sigma, 1.0)
    first = np.correlate(series, series[:m], mode='valid')

    profile = np.empty(n)
    qt = first.copy()
    for i in range(n):
        if i:
            qt[1:] = (qt[:-1] - series[:n - 1] * series[i - 1]
                      + series[m:m + n - 1] * series[i + m - 1])
            qt[0] = first[i]
        correlation = (qt - m * mean * mean[i]) / (m * safe * safe[i])
        distance = np.sqrt(np.maximum(2 * m * (1 - correlation), 0.0))
        lo, hi = max(0, i - exclusion), min(n, i + exclusion + 1)
        distance[lo:hi] = np.inf
        profile[i] = distance.min()
    return profile


def matrix_profile_scores(subset, window=STOMP_WINDOW):
    """Per-cycle matrix profile for one subset, aggregated over channels like the z-score.

    One profile per engine per channel, then the maximum over channels - the same aggregation
    the rolling z-score uses, so the two baselines differ in the statistic and not in how a
    multivariate series is reduced to one number.
    """
    _, evaluation = CMAPSS_DATA[subset]
    frame = evaluation.sort_values(['unit', 'cycle'])

    pieces = []
    for unit, engine in frame.groupby('unit', sort=True):
        values = engine[CHANNELS].to_numpy(dtype=float)
        n = len(values)
        if n < window + 2:
            continue
        best = np.zeros(n - window + 1)
        for channel in range(values.shape[1]):
            column = values[:, channel]
            if column.std() <= 1e-12:
                continue
            np.maximum(best, stomp(column, window), out=best)
        padded = np.full(n, np.nan)
        padded[window - 1:] = best
        pieces.append(pd.DataFrame({
            'unit': int(unit), 'end_rul': engine.rul.to_numpy(), 'profile': padded}))

    out = pd.concat(pieces, ignore_index=True)
    return out.dropna(subset=['profile']).reset_index(drop=True)


CURVES = {}
for subset in SUBSETS:
    CURVES[subset] = build_detector(subset)

    points = rolling_zscore(subset)
    healthy = points[points.end_rul > CORPUS_RUL_FLOOR]
    failing = points[points.end_rul <= HEALTH_HORIZON]
    CURVES[subset]['zscore'] = {
        'points': points,
        'curve': health_curve(points, column='zmax'),
        'separation': failing.zmax.median() / max(healthy.zmax.median(), 1e-9),
        'ranking': ranking_metrics(points.zmax.to_numpy(),
                                   points.end_rul.to_numpy() <= HEALTH_HORIZON),
    }

    mp = matrix_profile_scores(subset)
    mp_healthy = mp[mp.end_rul > CORPUS_RUL_FLOOR]
    mp_failing = mp[mp.end_rul <= HEALTH_HORIZON]
    CURVES[subset]['stomp'] = {
        'points': mp,
        'curve': health_curve(mp, column='profile'),
        'separation': mp_failing.profile.median() / max(mp_healthy.profile.median(), 1e-9),
        'ranking': ranking_metrics(mp.profile.to_numpy(),
                                   mp.end_rul.to_numpy() <= HEALTH_HORIZON),
    }

    slide = CURVES[subset]['slide']
    ours_healthy = slide[slide.end_rul > CORPUS_RUL_FLOOR]
    ours_failing = slide[slide.end_rul <= HEALTH_HORIZON]
    CURVES[subset]['ranking'] = ranking_metrics(
        slide.ratio.to_numpy(), slide.end_rul.to_numpy() <= HEALTH_HORIZON)
    CURVES[subset]['separation'] = (ours_failing.ratio.median()
                                    / max(ours_healthy.ratio.median(), 1e-9))

subset_table = pd.DataFrame([
    {'subset': subset, 'regime': REGIMES[subset],
     'corpus_windows': CURVES[subset]['counts']['corpus_windows'],
     'corpus_intervals': CURVES[subset]['counts']['corpus_intervals'],
     'rank': CURVES[subset]['counts']['rank'],
     'precision': CURVES[subset]['summary']['precision'],
     'recall': CURVES[subset]['summary']['recall'],
     'f1': CURVES[subset]['summary']['f1'],
     'adjusted_f1': CURVES[subset]['summary']['adjusted_f1'],
     'flagged': CURVES[subset]['summary']['flagged_fraction']}
    for subset in SUBSETS])

comparison = pd.DataFrame([
    {'subset': subset, 'regime': REGIMES[subset],
     'ours_roc_auc': CURVES[subset]['ranking'].get('roc_auc', float('nan')),
     'zscore_roc_auc': CURVES[subset]['zscore']['ranking'].get('roc_auc', float('nan')),
     'ours_lift': CURVES[subset]['ranking'].get('pr_auc_lift', float('nan')),
     'zscore_lift': CURVES[subset]['zscore']['ranking'].get('pr_auc_lift', float('nan')),
     'ours_separation': CURVES[subset]['separation'],
     'zscore_separation': CURVES[subset]['zscore']['separation'],
     'stomp_roc_auc': CURVES[subset]['stomp']['ranking'].get('roc_auc', float('nan')),
     'stomp_lift': CURVES[subset]['stomp']['ranking'].get('pr_auc_lift', float('nan')),
     'stomp_separation': CURVES[subset]['stomp']['separation']}
    for subset in SUBSETS])

RESULTS.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RESULTS / 'cmapss_baselines.csv', index=False)
pr_auc_rows = pd.DataFrame([
    {'subset': subset,
     'ours_pr_auc': CURVES[subset]['ranking'].get('pr_auc', float('nan')),
     'zscore_pr_auc': CURVES[subset]['zscore']['ranking'].get('pr_auc', float('nan')),
     'stomp_pr_auc': CURVES[subset]['stomp']['ranking'].get('pr_auc', float('nan'))}
    for subset in SUBSETS])
pr_auc_rows.to_csv(RESULTS / 'cmapss_baselines_prauc.csv', index=False)

display(subset_table.style.format({
    'corpus_windows': '{:,}', 'corpus_intervals': '{:,}',
    'precision': '{:.3f}', 'recall': '{:.3f}', 'f1': '{:.3f}',
    'adjusted_f1': '{:.3f}', 'flagged': '{:.2%}'}).hide(axis='index')
    .set_caption('signature detector'),
        comparison.style.format({
            'ours_roc_auc': '{:.4f}', 'zscore_roc_auc': '{:.4f}',
            'ours_lift': '{:.2f}', 'zscore_lift': '{:.2f}',
            'ours_separation': '{:.2f}', 'zscore_separation': '{:.2f}',
            'stomp_roc_auc': '{:.4f}', 'stomp_lift': '{:.2f}',
            'stomp_separation': '{:.2f}'}).hide(axis='index')
        .set_caption('signature detector against the rolling z-score baseline'))


In [ ]:
NEWLINE = chr(10)

FIGURES = PROJECT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

PANELS = (
    ('signature', None, '#1f77b4', 'signature detector',
     'score / threshold', True),
    ('zscore', 'zscore', '#7c3aed', 'rolling z-score baseline',
     'max |z| over channels', False),
    ('stomp', 'stomp', '#0f766e', 'STOMP matrix profile baseline',
     'matrix-profile distance', False),
)


def detector_curve(ax, subset, key, colour, is_ours):
    """One detector on one subset: interquartile band, mean and median, with the label bands."""
    curve = CURVES[subset]['curve'] if key is None else CURVES[subset][key]['curve']
    supported = curve[curve.n_engines >= MIN_ENGINES]
    band = ax.fill_between(supported.end_rul, supported.q25, supported.q75,
                           color=colour, alpha=0.18, lw=0, label='interquartile range')
    mean, = ax.plot(supported.end_rul, supported.mean_score, color=colour, lw=1.4,
                    label='mean')
    median, = ax.plot(supported.end_rul, supported.median_score, color='darkorange', lw=1.4,
                      label='median')
    degraded = ax.axvspan(0, HEALTH_HORIZON, color='crimson', alpha=0.12, lw=0,
                          label='labelled degraded')
    ambiguous = ax.axvspan(HEALTH_HORIZON, CORPUS_RUL_FLOOR, color='0.6', alpha=0.12, lw=0,
                           label='ambiguous')
    if is_ours:
        ax.set_yscale('log')
    ax.set_xlim(supported.end_rul.max(), 0)
    return [band, mean, median, degraded, ambiguous]


def detector_auc(subset, key):
    ranking = CURVES[subset]['ranking'] if key is None else CURVES[subset][key]['ranking']
    return ranking.get('roc_auc', float('nan'))


grid_files = []
for slug, key, colour, name, axis_label, is_ours in PANELS:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    for ax, subset in zip(axes.ravel(), SUBSETS):
        handles = detector_curve(ax, subset, key, colour, is_ours)
        ax.set_title('{0} - {1}   ROC-AUC {2:.3f}'.format(
            subset, REGIMES[subset], detector_auc(subset, key)), fontsize=9.5)

    for ax in axes[-1]:
        ax.set_xlabel("remaining useful life at the window's last cycle (cycles)")
    for ax in axes[:, 0]:
        ax.set_ylabel(axis_label if is_ours
                      else '{0} ({1}-cycle window)'.format(axis_label, ZSCORE_WINDOW))
    axes[0, 0].legend(handles=handles, fontsize=8, loc='upper left')
    fig.suptitle('C-MAPSS: {0}'.format(name), fontsize=13)
    plt.tight_layout()

    out = FIGURES / 'cmapss_grid_{0}.png'.format(slug)
    try:
        fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
        grid_files.append(out)
    except OSError:
        print('{0} was locked; not rewritten'.format(out.name))
    plt.show()

print('wrote {0} grid(s): {1}'.format(
    len(grid_files), ', '.join(p.name for p in grid_files)))

curve_files = []
for subset in SUBSETS:
    for slug, key, colour, name, axis_label, is_ours in PANELS:
        curve = (CURVES[subset]['curve'] if key is None else CURVES[subset][key]['curve'])
        supported = curve[curve.n_engines >= MIN_ENGINES]
        ranking = (CURVES[subset]['ranking'] if key is None
                   else CURVES[subset][key]['ranking'])

        single, ax = plt.subplots(figsize=(8.5, 4.4))
        ax.fill_between(supported.end_rul, supported.q25, supported.q75,
                        color=colour, alpha=0.18, lw=0, label='interquartile range')
        ax.plot(supported.end_rul, supported.mean_score, color=colour, lw=1.5, label='mean')
        ax.plot(supported.end_rul, supported.median_score, color='darkorange', lw=1.5,
                label='median')
        ax.axvspan(0, HEALTH_HORIZON, color='crimson', alpha=0.12, lw=0,
                   label='labelled degraded')
        ax.axvspan(HEALTH_HORIZON, CORPUS_RUL_FLOOR, color='0.6', alpha=0.12, lw=0,
                   label='ambiguous')
        if is_ours:
            ax.set_yscale('log')
        ax.set_xlim(supported.end_rul.max(), 0)
        ax.set_xlabel("remaining useful life at the window's last cycle (cycles)")
        ax.set_ylabel(axis_label if is_ours
                      else '{0} ({1}-cycle window)'.format(axis_label, ZSCORE_WINDOW))
        ax.legend(fontsize=7.5, loc='upper left')
        ax.set_title('{0} - {1}'.format(subset, REGIMES[subset]) + NEWLINE
                     + '{0}, ROC-AUC {1:.3f}'.format(
                         name, ranking.get('roc_auc', float('nan'))),
                     fontsize=10)
        single.tight_layout()

        out = FIGURES / 'cmapss_{0}_{1}.png'.format(subset.lower(), slug)
        try:
            single.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
            curve_files.append(out)
        except OSError:
            print('{0} was locked; not rewritten'.format(out.name))
        plt.close(single)

print('wrote {0} figure(s) across {1} subsets and {2} detectors'.format(
    len(curve_files), len(SUBSETS), len(PANELS)))

rows = []
for subset in SUBSETS:
    slide, summary = CURVES[subset]['slide'], CURVES[subset]['summary']
    healthy = slide[slide.end_rul > CORPUS_RUL_FLOOR]
    failing = slide[slide.end_rul <= HEALTH_HORIZON]
    ambiguous = slide[(slide.end_rul > HEALTH_HORIZON)
                      & (slide.end_rul <= CORPUS_RUL_FLOOR)]
    rows.append({
        'subset': subset, 'regime': REGIMES[subset],
        'rank': CURVES[subset]['counts']['rank'],
        'sep_median': failing.ratio.median() / max(healthy.ratio.median(), 1e-9),
        'degraded>1': float((failing.ratio > 1.0).mean()),
        'ambiguous>1': float((ambiguous.ratio > 1.0).mean()),
        'healthy>1': float((healthy.ratio > 1.0).mean()),
        'precision': summary['precision'], 'recall': summary['recall'],
        'f1': summary['f1'], 'adjusted_f1': summary['adjusted_f1'],
    })

display(pd.DataFrame(rows).style.format(precision=3).hide(axis='index')
        .set_caption('separation = degraded median / healthy median, '
                     'both as score/threshold'))

The four subsets fall into two pairs. FD001 and FD003 have **one** operating condition; FD002
and FD004 have **six**, and every engine visits all of them - a single 17-cycle window spans
5.7 of the six on average, so the condition varies *inside* the window rather than between
engines.
This likely demonstrates the much lower performance on these data sets.

## 3. SMD

In [ ]:
%%capture
%%time
%%capture
import gc
import time

from anomalies_scale.AUC import (
    POINT_SCORE_COLUMN, ranking_metrics, score_points)
from anomalies_scale.index_creation import PooledIndex
from anomalies_scale.anomaly_detection_pooled import (
    PooledCorpus, build_whitening, detect_anomalies, split_corpus)
from anomalies_scale.interval_detection import detect_anomalous_intervals, parse_statistic
from anomalies_scale.covariance_creation import signature_matrix
from anomalies_scale.signature_computer import compute_corpus
from anomalies_scale.canonical_streams import to_canonical
from anomalies_scale.stream_evaluation import segments
from anomalies_scale.throughput import peak_memory_bytes

SMD_ROOT = PROJECT / 'OmniAnomaly-master' / 'ServerMachineDataset'
SMD_MACHINES = sorted(p.stem for p in (SMD_ROOT / 'train').glob('*.txt'))

SMD_WINDOW, SMD_STRIDE = 100, 50
SMD_TRUNC, SMD_GRANULARITY = 2, 3
SMD_BAND = 1
SMD_SIG_TOL, SMD_TOL = 2, 2
SMD_FIT_FRACTION = 0.8
SMD_PERCENTILE = 99.9
SMD_VARIANCE_KEEP = 0.999
SMD_UMAP_DIM = 10
SMD_TRUNC_C = 3
SMD_TRUNC_D = 3
SMD_DRAWS = 30
SMD_FOLDS = CONFIG['calibrate']['folds']


def smd_load(name):
    """One machine's (train, test, labels). Train is unlabelled and assumed anomaly-free."""
    read = lambda kind: pd.read_csv(SMD_ROOT / kind / (name + '.txt'), header=None)
    return read('train'), read('test'), read('test_label')[0].to_numpy().astype(bool)


def smd_windows(frame):
    """Overlapping windows as a (n, SMD_WINDOW, width+1) array, time in the first channel.

    Arrays rather than the parquet-per-window the SMD notebook wrote: `detect_anomalies` takes
    either, and 28 machines would otherwise mean about 31,000 files on disk for no gain.
    """
    values = frame.to_numpy(dtype=float)
    starts = range(0, len(values) - SMD_WINDOW + 1, SMD_STRIDE)
    clock = np.linspace(0.0, 1.0, SMD_WINDOW).reshape(-1, 1)
    blocks = [np.hstack([clock, values[lo:lo + SMD_WINDOW]]) for lo in starts]
    return np.stack(blocks), list(starts)


def smd_canonical(paths, stem, groups=1):
    """Window arrays as a canonical stream set, for `compute_corpus` and `run_bagged`.

    `groups` exists for the calibration inside `run_bagged`. `crossvalidated_thresholds`
    forms its folds over *source* streams, and `source_of` strips a trailing `_N` - so naming
    every window `stem_00001` collapses all of them onto one source and the fold split fails
    with "the corpus has 1". Splitting the windows into contiguous groups gives it something
    to hold out, and contiguous rather than interleaved so a held-out fold is a held-out
    period rather than a scatter of neighbours.
    """
    columns = ['ch{0}'.format(i) for i in range(paths.shape[2] - 1)]
    edges = np.linspace(0, len(paths), int(groups) + 1, dtype=int)
    group_of = np.zeros(len(paths), dtype=int)
    for g in range(int(groups)):
        group_of[edges[g]:edges[g + 1]] = g

    rows = []
    for i, block in enumerate(paths):
        frame = pd.DataFrame(block[:, 1:], columns=columns)
        frame.insert(0, 'time', block[:, 0])
        name = ('{0}_{1:05d}'.format(stem, i) if groups == 1
                else '{0}g{1:02d}_{2:05d}'.format(stem, group_of[i], i))
        rows.append((name, frame))
    return to_canonical(rows, time_col='time')


def smd_prepare(name):
    """Window one machine into fit / withheld / test, keeping every channel."""
    train, test, labels = smd_load(name)
    fit_all, base_starts = smd_windows(train)
    test_paths, test_starts = smd_windows(test)
    cut = int(round(SMD_FIT_FRACTION * len(fit_all)))
    return {
        'name': name, 'labels': labels, 'n_points': len(labels),
        'fit_paths': fit_all[:cut], 'held_paths': fit_all[cut:], 'test_paths': test_paths,
        'test_starts': test_starts, 'width': fit_all.shape[2],
        'test_frame': test,
    }


def smd_series(frame):
    """The whole test series as one canonical stream, on the windows' time scale.

    `score_points` slides its own window, so it wants the series rather than the windows - and
    the time channel has to advance at the same rate the corpus windows do, or a 100-point
    slice of this spans a different amount of "time" than a corpus window and its signature is
    not comparable. One window is 1.0 of time, so the step is 1 / (SMD_WINDOW - 1).
    """
    values = frame.to_numpy(dtype=float)
    block = pd.DataFrame(values, columns=['ch{0}'.format(i) for i in range(values.shape[1])])
    block.insert(0, 'time', np.arange(len(values)) / (SMD_WINDOW - 1))
    return to_canonical([('test', block)], time_col='time')


def smd_engine(corpus, whitening):
    """A `PooledIndex` over the same corpus and metric the pooled detector searches."""
    signatures, terms = signature_matrix(corpus)
    depths = corpus['depth'].to_numpy(dtype=int)
    whitened = np.ascontiguousarray(signatures @ np.asarray(whitening).T, dtype='float32')
    index = faiss.IndexFlatL2(whitened.shape[1])
    index.add(whitened)
    return PooledIndex(index, depth=depths, split=np.zeros(len(depths), dtype=int),
                       band=SMD_BAND, terms=terms)


def smd_point_scores(frame, corpus, whitening, threshold, trunc):
    """A continuous score for every point of the test series, independent of the threshold.

    This is what makes the AUC column mean what it says. Rasterising `detect_anomalies`'
    output cannot: that function returns only intervals that *exceeded* the threshold, so
    every unflagged point scores zero and the "ranking" is mostly ties at zero - 73% of them
    for the pooled detector. `score_points` scores every point on a fixed sliding window
    whether or not anything fired, so raising or lowering the threshold cannot move it.
    """
    scored = score_points(smd_series(frame), smd_engine(corpus, whitening), whitening,
                          threshold, window=SMD_WINDOW, stride=SMD_STRIDE,
                          span=SMD_WINDOW, trunc=trunc, neighbours=1)
    return np.concatenate(list(scored[POINT_SCORE_COLUMN]))


def smd_rasterise(flagged, starts, n_points, score_column='score'):
    """Flagged intervals to a boolean mask and a per-point score.

    The score is the largest any window assigns to a point, which is what makes an ROC over
    overlapping windows well defined - a point covered by two windows takes the stronger claim.
    """
    mask = np.zeros(n_points, dtype=bool)
    score = np.zeros(n_points, dtype=float)
    for row in flagged.itertuples(index=False):
        index = int(str(row.stream).rsplit('_', 1)[-1])
        offset = starts[index] if index < len(starts) else 0
        lo, hi = offset + int(row.start_index), offset + int(row.end_index) + 1
        mask[lo:hi] = True
        value = float(getattr(row, score_column, 1.0))
        np.maximum(score[lo:hi], value, out=score[lo:hi])
    return mask, score


def smd_interval_scores(pooled, frame, exclude_self=False):
    """Nearest-neighbour distance for every interval of a corpus frame, one query per depth.

    `PooledCorpus.score` takes a single interval, and calibration needs tens of thousands, so
    this queries each depth's index once instead. It reaches into `pooled._index`, which is
    private - the module has no batched scorer, and the SMD notebook made the same note.

    `exclude_self` matters when the frame's own rows are in the index: each is then its own
    nearest neighbour at distance zero, so the second neighbour is the meaningful one.
    """
    signatures, depths = split_corpus(frame)
    out = np.empty(len(signatures))
    k = 2 if exclude_self else 1
    for depth in np.unique(depths):
        rows = np.flatnonzero(depths == depth)
        queries = np.ascontiguousarray(
            (signatures[rows] @ pooled.covariance.T).astype('float32'))
        distances, _ = pooled._index(int(depth)).search(queries, k)
        out[rows] = distances[:, k - 1]
    return out


class RssSampler:
    """Maximum resident memory observed while the block runs, in bytes.

    `peak_memory_bytes` cannot answer this: on Windows it reports `peak_wset`, the process's
    lifetime high-water mark, which never falls and so attributes every later reading to
    whichever detector happened to set the record. Polling RSS on a thread and keeping the
    maximum gives a figure that belongs to one detector and can be compared across them.

    The baseline is subtracted into `used` - what the detector added on top of an already
    loaded kernel - while `peak` stays absolute, because both are worth reporting: `used` is
    the algorithm's appetite, `peak` is what the machine has to supply.
    """

    def __init__(self, interval=0.25):
        self.interval = interval
        self.peak = self.baseline = 0
        self._stop = None
        self._thread = None

    def _watch(self):
        import psutil

        process = psutil.Process()
        while not self._stop.wait(self.interval):
            try:
                self.peak = max(self.peak, int(process.memory_info().rss))
            except Exception:
                return

    def __enter__(self):
        import threading

        import psutil

        self.baseline = self.peak = int(psutil.Process().memory_info().rss)
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._watch, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *_):
        self._stop.set()
        self._thread.join(timeout=2.0)
        return False

    @property
    def used(self):
        """Bytes added on top of the baseline - the detector's own appetite."""
        return max(self.peak - self.baseline, 0)


def smd_metrics(mask, score, truth, seconds, points_scored, label):
    """Every metric this section reports, for one detector on one machine."""
    tp = int((mask & truth).sum())
    fp = int((mask & ~truth).sum())
    fn = int((~mask & truth).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    adjusted = mask.copy()
    for lo, hi in segments(truth):
        if adjusted[lo:hi].any():
            adjusted[lo:hi] = True
    adj_tp = int((adjusted & truth).sum())
    adj_fp = int((adjusted & ~truth).sum())
    adj_precision = adj_tp / (adj_tp + adj_fp) if adj_tp + adj_fp else 0.0
    adj_recall = adj_tp / max(int(truth.sum()), 1)
    adj_f1 = (2 * adj_precision * adj_recall / (adj_precision + adj_recall)
              if adj_precision + adj_recall else 0.0)

    record = {'detector': label, 'precision': precision, 'recall': recall, 'f1': f1,
              'adj_precision': adj_precision, 'adj_recall': adj_recall, 'adj_f1': adj_f1,
              'flagged': float(mask.mean()), 'anomaly_rate': float(truth.mean()),
              'seconds': seconds, 'points_per_second': points_scored / max(seconds, 1e-9),
              'peak_rss_mb': peak_memory_bytes() / 1e6}
    record.update(ranking_metrics(score, truth) if np.ptp(score) > 0 else {})
    return record


print('{0} machines, {1} channels, window {2} stride {3}'.format(
    len(SMD_MACHINES), smd_load(SMD_MACHINES[0])[0].shape[1], SMD_WINDOW, SMD_STRIDE))
probe = smd_prepare(SMD_MACHINES[0])
display(summarise(machines=len(SMD_MACHINES), width=probe['width'],
                  fit_windows=len(probe['fit_paths']),
                  held_windows=len(probe['held_paths']),
                  test_windows=len(probe['test_paths']),
                  test_points=probe['n_points'],
                  anomaly_rate=round(float(probe['labels'].mean()), 4),
                  signature_terms=iisignature.siglength(probe['width'], SMD_TRUNC)))
del probe
gc.collect()

### 3.1 The four detectors

In [ ]:
%%capture
%%time
%%capture

def smd_calibrate(fit_paths, held_paths, stem, trunc=SMD_TRUNC):
    """Corpus, whitening and threshold for one machine, by the three-step protocol."""
    corpus_fit = compute_corpus(smd_canonical(fit_paths, stem + '_fit'),
                                trunc=trunc, granularity=SMD_GRANULARITY)
    corpus_held = compute_corpus(smd_canonical(held_paths, stem + '_held'),
                                 trunc=trunc, granularity=SMD_GRANULARITY)
    corpus_final = pd.concat([corpus_fit, corpus_held], ignore_index=True)

    signatures_fit, _ = split_corpus(corpus_fit)
    centred = (np.asarray(signatures_fit, float)
               - np.asarray(signatures_fit, float).mean(axis=0))
    factor = np.linalg.qr(centred, mode='r') if centred.shape[0] > centred.shape[1] else centred
    values = np.linalg.svd(factor, compute_uv=False)
    energy = np.cumsum(values ** 2) / np.sum(values ** 2)
    rank = min(int(np.searchsorted(energy, SMD_VARIANCE_KEEP) + 1),
               int((values > 1e-12 * values.max()).sum()))
    rcond = float(values[rank - 1] / values[0]) * (1 - 1e-9)
    whitening = build_whitening(signatures_fit, rcond=rcond)

    pooled_final = PooledCorpus(corpus_final, covariance=whitening, band=SMD_BAND,
                                check_scale=False)
    scores = smd_interval_scores(pooled_final, corpus_held, exclude_self=True)
    threshold = float(np.percentile(scores, SMD_PERCENTILE))
    return corpus_final, whitening, threshold, rank


def detector_pooled(spec, label='A pooled', paths_key='test_paths', prepared=None,
                    trunc=SMD_TRUNC):
    """Detector A: widest-first search over one pooled corpus and one fitted metric."""
    started = time.perf_counter()
    corpus, whitening, threshold, rank = prepared or smd_calibrate(
        spec['fit_paths'], spec['held_paths'], spec['name'])
    flagged = detect_anomalies(list(spec[paths_key]), corpus, threshold=threshold,
                               covariance=whitening, band=SMD_BAND,
                               sig_tol=SMD_SIG_TOL, tol=SMD_TOL)
    mask, _ = smd_rasterise(flagged, spec['test_starts'], spec['n_points'])
    score = smd_point_scores(spec.get('score_frame', spec['test_frame']), corpus, whitening,
                             threshold, trunc)[:spec['n_points']]
    return mask, score, time.perf_counter() - started, {'metric_rank': rank}


def detector_per_interval(spec, label='B per interval'):
    """Detector B: corpus and metric rebuilt for every interval the search visits."""
    started = time.perf_counter()
    names = ['stream_{0}'.format(i) for i in range(len(spec['test_paths']))]
    normal = np.concatenate([spec['fit_paths'], spec['held_paths']])
    table, stats = detect_anomalous_intervals(
        normal, spec['test_paths'], names, trunc=SMD_TRUNC,
        sig_tol=SMD_SIG_TOL, tol=SMD_TOL, variance_keep=SMD_VARIANCE_KEEP,
        statistic=parse_statistic('p{0}'.format(SMD_PERCENTILE)))
    flagged = table[table.exceeds_threshold] if 'exceeds_threshold' in table else table
    mask, _ = smd_rasterise(flagged, spec['test_starts'], spec['n_points'])
    _, score = smd_rasterise(table, spec['test_starts'], spec['n_points'],
                             score_column='ratio')
    return mask, score, time.perf_counter() - started, {
        'metric_rank': float(stats.get('mean_rank', float('nan'))),
        'intervals_built': int(stats.get('intervals_built', 0))}


def detector_umap(spec, label='C UMAP'):
    """Detector C: the pooled detector on a UMAP reduction of the channels."""
    from anomalies_scale.umap_projection import fit_embedding

    started = time.perf_counter()
    ambient = np.vstack([block[:, 1:] for block in spec['fit_paths']])
    projection, info = fit_embedding(ambient, SMD_UMAP_DIM, context=1)

    def reduce(paths):
        clock = paths[0][:, :1]
        return np.stack([np.hstack([clock, projection.transform(b[:, 1:])]) for b in paths])

    latent = {k: reduce(spec[k]) for k in ('fit_paths', 'held_paths', 'test_paths')}
    inner = dict(spec, **latent)
    inner['score_frame'] = pd.DataFrame(
        projection.transform(spec['test_frame'].to_numpy(dtype=float)))
    prepared = smd_calibrate(inner['fit_paths'], inner['held_paths'],
                             spec['name'] + '_umap', trunc=SMD_TRUNC_C)
    mask, score, _, extra = detector_pooled(inner, prepared=prepared, trunc=SMD_TRUNC_C)
    extra['trustworthiness'] = float(info.get('trustworthiness', float('nan')))
    return mask, score, time.perf_counter() - started, extra


def detector_bagged(spec, label='D bagged'):
    """Detector D: the pooled search over random channel subsets, votes unioned."""
    from anomalies_scale.bagging import run_bagged

    started = time.perf_counter()
    corpus_streams = smd_canonical(np.concatenate([spec['fit_paths'], spec['held_paths']]),
                                   spec['name'] + '_corpus', groups=SMD_FOLDS + 1)
    test_streams = smd_canonical(spec['test_paths'], spec['name'] + '_test')
    scored, votes, diagnostics = run_bagged(
        corpus_streams, test_streams, draws=SMD_DRAWS, trunc=SMD_TRUNC_D,
        granularity=SMD_GRANULARITY, variance_keep=SMD_VARIANCE_KEEP, band=SMD_BAND,
        statistic='p{0}'.format(SMD_PERCENTILE), sig_tol=SMD_SIG_TOL,
        tol=SMD_TOL, folds=SMD_FOLDS, votes=1)

    mask = np.zeros(spec['n_points'], dtype=bool)
    score = np.zeros(spec['n_points'], dtype=float)
    fraction = dict(zip(votes.stream, votes.votes)) if 'votes' in votes else {}
    for i, name in enumerate(scored.stream):
        offset = spec['test_starts'][i] if i < len(spec['test_starts']) else 0
        for lo, hi in scored.anomalous.iloc[i]:
            mask[offset + lo:offset + hi + 1] = True
        share = np.asarray(fraction.get(name, []), dtype=float)
        if share.size:
            end = min(offset + share.size, spec['n_points'])
            np.maximum(score[offset:end], share[:end - offset], out=score[offset:end])
    if np.ptp(score) == 0:
        score = mask.astype(float)
    return mask, score, time.perf_counter() - started, {
        'mean_rank': float(diagnostics.get('mean_rank', float('nan'))),
        'draws': SMD_DRAWS}


SMD_DETECTORS = [('A pooled', detector_pooled),
                 ('B per interval', detector_per_interval),
                 ('C UMAP', detector_umap),
                 ('D bagged', detector_bagged)]
print('{0} detectors defined: {1}'.format(
    len(SMD_DETECTORS), ', '.join(name for name, _ in SMD_DETECTORS)))

### 3.2 Every machine, every detector

In [ ]:
%%capture
%%time
%%capture
SMD_LIMIT = None

SMD_CSV = RESULTS / 'smd_detectors.csv'
smd_rows = (pd.read_csv(SMD_CSV).to_dict('records')
            if SMD_CSV.exists() else [])
done = {(r['machine'], r['detector']) for r in smd_rows}
if done:
    print('resuming: {0} row(s) already on disk'.format(len(smd_rows)))
smd_started = time.perf_counter()
machines = SMD_MACHINES[:SMD_LIMIT] if SMD_LIMIT else SMD_MACHINES

for number, machine in enumerate(machines, start=1):
    spec = smd_prepare(machine)
    truth = spec['labels']
    shared = smd_calibrate(spec['fit_paths'], spec['held_paths'], machine)

    for label, detector in SMD_DETECTORS:
        if (machine, label) in done:
            continue
        try:
            with RssSampler() as memory:
                if label == 'A pooled':
                    mask, score, seconds, extra = detector(spec, prepared=shared)
                else:
                    mask, score, seconds, extra = detector(spec)
        except Exception as error:
            print('  {0:<16} {1}: {2}'.format(label, type(error).__name__, error))
            continue
        row = smd_metrics(mask, score, truth, seconds, spec['n_points'], label)
        row.update(machine=machine, **{k: v for k, v in extra.items()})
        row['process_peak_rss_mb'] = row.get('peak_rss_mb')
        row['peak_rss_mb'] = memory.peak / (1024 * 1024)
        row['used_rss_mb'] = memory.used / (1024 * 1024)
        smd_rows.append(row)

    RESULTS.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(smd_rows).to_csv(SMD_CSV, index=False)
    finished = [r for r in smd_rows if r['machine'] == machine]
    print('{0:>2}/{1} {2:<14} '.format(number, len(machines), machine)
          + '  '.join('{0} F1 {1:.3f}/AUC {2:.3f}'.format(
              r['detector'].split()[0], r['f1'], r.get('roc_auc', float('nan')))
              for r in finished), flush=True)
    del spec, shared
    gc.collect()

smd = pd.DataFrame(smd_rows)
smd_seconds = time.perf_counter() - smd_started
smd.to_csv(SMD_CSV, index=False)
print('wrote {0}'.format(SMD_CSV))

COLUMNS = ['machine', 'detector', 'anomaly_rate', 'precision', 'recall', 'f1',
           'adj_precision', 'adj_recall', 'adj_f1', 'roc_auc', 'pr_auc', 'pr_auc_lift',
           'flagged', 'seconds', 'points_per_second', 'peak_rss_mb', 'used_rss_mb']
present = [c for c in COLUMNS if c in smd.columns]

by_detector = smd.groupby('detector').agg(
    machines=('machine', 'nunique'), f1=('f1', 'mean'), adj_f1=('adj_f1', 'mean'),
    roc_auc=('roc_auc', 'mean'), pr_auc_lift=('pr_auc_lift', 'mean'),
    flagged=('flagged', 'mean'), seconds=('seconds', 'mean'),
    points_per_second=('points_per_second', 'mean'), peak_rss_mb=('peak_rss_mb', 'max'), used_rss_mb=('used_rss_mb', 'mean'))

display(by_detector.style.format({
    'f1': '{:.4f}', 'adj_f1': '{:.4f}', 'roc_auc': '{:.4f}', 'pr_auc_lift': '{:.2f}',
    'flagged': '{:.2%}', 'seconds': '{:.1f}', 'points_per_second': '{:,.0f}',
    'peak_rss_mb': '{:,.0f}', 'used_rss_mb': '{:,.0f}'}).set_caption(
        'mean over {0} machine(s); {1:.0f}s total'.format(smd.machine.nunique(),
                                                          smd_seconds)),
        smd[present].sort_values(['machine', 'detector'])
        .style.format(precision=4).hide(axis='index')
        .set_caption('every machine, every detector'))

### 3.3 Where each detector marks

In [ ]:
%%time
SHOW_MACHINES = ['machine-2-8', 'machine-1-5', 'machine-1-6', 'machine-3-9']
SHOW_CHANNELS = 3
SMD_MARK_CACHE = RESULTS / 'smd_marks'
NEWLINE = chr(10)

DETECTOR_COLOUR = {'A pooled': '#1d4ed8', 'B per interval': '#b45309',
                   'C UMAP': '#6d28d9', 'D bagged': '#b91c1c'}
TRUTH_COLOUR = '#dc2626'


def detector_key(label):
    """'A pooled' -> 'A', for cache keys and short row labels."""
    return label.split()[0]


def youden_cut(score, truth):
    """The ROC point maximising TPR - FPR, as a threshold and the mask it implies."""
    from sklearn.metrics import roc_curve

    truth = np.asarray(truth, dtype=bool)
    score = np.where(np.isfinite(score), np.asarray(score, dtype=float), 0.0)
    if truth.all() or not truth.any() or np.ptp(score) == 0:
        return float('nan'), np.zeros(score.shape, dtype=bool)
    false_rate, true_rate, cuts = roc_curve(truth, score)
    cut = float(cuts[int(np.argmax(true_rate - false_rate))])
    return cut, score >= cut


def f1_cut(score, truth):
    """The cut maximising F1 - the same scores under a criterion that charges for precision."""
    from sklearn.metrics import precision_recall_curve

    truth = np.asarray(truth, dtype=bool)
    score = np.where(np.isfinite(score), np.asarray(score, dtype=float), 0.0)
    if truth.all() or not truth.any() or np.ptp(score) == 0:
        return float('nan'), np.zeros(score.shape, dtype=bool)
    precision, recall, cuts = precision_recall_curve(truth, score)
    harmonic = 2 * precision[:-1] * recall[:-1] / np.maximum(
        precision[:-1] + recall[:-1], 1e-12)
    if not len(harmonic):
        return float('nan'), np.zeros(score.shape, dtype=bool)
    cut = float(cuts[int(np.nanargmax(harmonic))])
    return cut, score >= cut


def cut_quality(mask, truth):
    """Precision, recall and F1 of one mask against the point labels."""
    from sklearn.metrics import precision_recall_fscore_support

    precision, recall, f1, _ = precision_recall_fscore_support(
        np.asarray(truth, dtype=bool), np.asarray(mask, dtype=bool),
        average='binary', zero_division=0)
    return float(precision), float(recall), float(f1)


def smd_marks(machine):
    """Each detector's calibrated mask and continuous score, from cache when available."""
    spec = smd_prepare(machine)
    cache = SMD_MARK_CACHE / '{0}.npz'.format(machine)

    if cache.exists():
        stored = np.load(cache)
        results = {label: {'score': stored['score_' + detector_key(label)],
                           'calibrated': stored['mask_' + detector_key(label)].astype(bool)}
                   for label, _ in SMD_DETECTORS
                   if 'score_' + detector_key(label) in stored}
        print('  cached scores for {0} detector(s)'.format(len(results)))
        return spec, results

    shared = smd_calibrate(spec['fit_paths'], spec['held_paths'], machine)
    results, payload = {}, {}
    for label, detector in SMD_DETECTORS:
        try:
            if label == 'A pooled':
                mask, score, _, _ = detector(spec, prepared=shared)
            else:
                mask, score, _, _ = detector(spec)
        except Exception as error:
            print('  {0:<16} {1}: {2}'.format(label, type(error).__name__, error))
            continue
        mask = np.asarray(mask, dtype=bool)
        score = np.asarray(score, dtype=float)
        results[label] = {'calibrated': mask, 'score': score}
        payload['score_' + detector_key(label)] = score
        payload['mask_' + detector_key(label)] = mask

    SMD_MARK_CACHE.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache, labels=np.asarray(spec['labels'], dtype=bool), **payload)
    print('  cached to {0}'.format(cache.name))
    return spec, results


def resolve_cuts(spec, results):
    """Add both cuts to every detector's entry, and tabulate them beside the calibrated one."""
    truth = np.asarray(spec['labels'], dtype=bool)
    rows = []
    for label, _ in SMD_DETECTORS:
        entry = results.get(label)
        if entry is None:
            continue
        cut, mask = youden_cut(entry['score'], truth)
        entry['cut'], entry['oracle'] = cut, mask
        precision, recall, f1 = cut_quality(mask, truth)
        entry['f1'] = f1

        best_cut, best_mask = f1_cut(entry['score'], truth)
        entry['f1_cut'], entry['f1_mask'] = best_cut, best_mask
        best_quality = cut_quality(best_mask, truth)

        rows.append({
            'machine': spec['name'], 'detector': label,
            'roc_auc': ranking_metrics(entry['score'], truth).get('roc_auc', float('nan')),
            'anomaly_rate': float(truth.mean()),
            'roc_flagged': float(mask.mean()), 'roc_precision': precision,
            'roc_recall': recall, 'roc_f1': f1,
            'f1cut_flagged': float(best_mask.mean()), 'f1cut_precision': best_quality[0],
            'f1cut_recall': best_quality[1], 'f1cut_f1': best_quality[2],
            'calib_flagged': float(entry['calibrated'].mean()),
            'calib_f1': cut_quality(entry['calibrated'], truth)[2]})
    return pd.DataFrame(rows)


def plot_marks(machine, spec, results, detectors=None, suffix='detectors'):
    """Context channels with the truth shaded, and one band per detector beneath it."""
    detectors = [d for d in (detectors or [label for label, _ in SMD_DETECTORS])
                 if d in results]
    truth = np.asarray(spec['labels'], dtype=bool)
    frame = spec['test_frame']
    order = frame.std().sort_values(ascending=False).index[:SHOW_CHANNELS]
    x = np.arange(len(truth))
    spans = list(segments(truth))

    fig, (top, bottom) = plt.subplots(
        2, 1, figsize=(15, 5.6), sharex=True,
        gridspec_kw={'height_ratios': [2.2, 0.32 + 0.34 * (len(detectors) + 1)],
                     'hspace': 0.12})

    for offset, column in enumerate(order):
        values = frame[column].to_numpy(dtype=float)
        spread = values.std() or 1.0
        top.plot(x, (values - values.mean()) / spread + offset * 4.5, lw=0.7,
                 color='#334155')
    for lo, hi in spans:
        top.axvspan(lo, hi, color=TRUTH_COLOUR, alpha=0.30, lw=0)
        bottom.axvspan(lo, hi, color=TRUTH_COLOUR, alpha=0.18, lw=0)
    top.set_ylabel('{0} most variable channels'.format(SHOW_CHANNELS) + NEWLINE
                   + '(standardised, offset)', fontsize=8)
    top.set_yticks([])
    top.spines[['top', 'right', 'left']].set_visible(False)

    rows = [('labelled anomalies', truth, TRUTH_COLOUR)]
    rows += [(label, results[label]['oracle'], DETECTOR_COLOUR[label]) for label in detectors]

    for position, (label, mask, colour) in enumerate(rows):
        y = len(rows) - position - 1
        bottom.fill_between(x, y + 0.12, y + 0.88, where=mask, step='mid',
                            color=colour, lw=0)
        share = float(mask.mean())
        text = ('{0}   {1:.1%}'.format(label, share) if position == 0
                else '{0}   {1:.1%} flagged   F1 {2:.2f}'.format(
                    label, share, results[label]['f1']))
        bottom.text(-len(truth) * 0.012, y + 0.5, text, fontsize=8, ha='right',
                    va='center', color=colour,
                    fontweight='bold' if position == 0 else 'normal')
    bottom.set_ylim(0, len(rows))
    bottom.set_yticks([])
    bottom.set_xlim(0, len(truth))
    bottom.set_xlabel('test-series point index')
    bottom.spines[['top', 'right', 'left']].set_visible(False)

    top.text(0.0, 1.02, '{0}   shaded red: labelled anomalies'.format(machine),
             transform=top.transAxes, ha='left', va='bottom', fontsize=11)
    plt.tight_layout()

    FIGURES = PROJECT / 'figures'
    FIGURES.mkdir(parents=True, exist_ok=True)
    out = FIGURES / 'smd_{0}_{1}.png'.format(machine, suffix)
    try:
        fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    except OSError:
        print('  {0} was locked; not rewritten'.format(out.name))
    plt.show()
    return out


smd_mark_files, smd_cut_tables = [], []
for machine in SHOW_MACHINES:
    print(machine)
    spec, results = smd_marks(machine)
    smd_cut_tables.append(resolve_cuts(spec, results))
    smd_mark_files.append(plot_marks(machine, spec, results))
    smd_mark_files.append(plot_marks(machine, spec, results,
                                     detectors=['A pooled', 'B per interval'], suffix='ab'))
    del spec, results
    gc.collect()

smd_cuts = pd.concat(smd_cut_tables, ignore_index=True)
display(smd_cuts.style.format({
    'roc_auc': '{:.3f}', 'anomaly_rate': '{:.1%}',
    'roc_flagged': '{:.1%}', 'roc_precision': '{:.3f}', 'roc_recall': '{:.3f}',
    'roc_f1': '{:.3f}', 'f1cut_flagged': '{:.1%}', 'f1cut_precision': '{:.3f}',
    'f1cut_recall': '{:.3f}', 'f1cut_f1': '{:.3f}',
    'calib_flagged': '{:.1%}', 'calib_f1': '{:.3f}'}).hide(axis='index')
    .set_caption('ROC-optimal and F1-optimal cuts, with the calibrated mask beside them'))
print('{0} figure(s) written'.format(len(smd_mark_files)))


## 4. Exathlon and TimeEval
This section aims to explore how the detector scales up with both dimension. We aim to answer the questions of whether higher dimensional data improves detector accuracy, or hinders it due to the increase in computational load required to handle data of this size. 
### 7.1 Loading the traces and their labels

In [ ]:
%%capture
import gc
import time

import faiss

from anomalies_scale.canonical_streams import as_matrix, to_canonical
from anomalies_scale.normalisation import fit_normaliser
from anomalies_scale.covariance_creation import (
    Whitening, retained_rank, signature_matrix, spectrum)
from anomalies_scale.crossvalidated_thresholding import crossvalidated_thresholds
from anomalies_scale.signature_computer import compute_corpus
from anomalies_scale.stream_evaluation import (
    evaluate_predictions, intervals_to_mask, prf, segments)
from anomalies_scale.throughput import peak_memory_bytes
from anomalies_scale.windowing import window_streams

EXATHLON = PROJECT / 'data' / 'exathlon'
EXATHLON_PARQUET = EXATHLON / 'parquet'
EXATHLON_TRUNC = 2
NEIGHBOURS = CONFIG['detect']['neighbours']
STATISTIC = CONFIG['calibrate']['statistic']
FOLDS = CONFIG['calibrate']['folds']
BAND = CONFIG['detect']['band']
GRANULARITY = CONFIG['signature']['granularity']
VARIANCE_KEEP = CONFIG['metric']['variance_keep']

NORMALISE = CONFIG['signature']['normalise_method']
RCOND = CONFIG['metric'].get('rcond')

UMAP_DIMENSION = 47
WINDOW_TRUNC = 2
NULL_FILL = 0.0
NEWLINE = chr(10)

CHANNELS_ALL = json.loads(
    (EXATHLON_PARQUET / 'channels.json').read_text(encoding='utf-8'))['columns']
VALUE_CHANNELS = [c for c in CHANNELS_ALL if c != 't']

exathlon_traces = pd.DataFrame([
    {'trace': path.stem, 'side': side, 'path': path,
     'app_id': int(path.stem.split('_')[0]), 'type_id': int(path.stem.split('_')[1]),
     'input_rate': int(path.stem.split('_')[2]), 'trace_id': int(path.stem.split('_')[3]),
     'bytes': path.stat().st_size}
    for side in ('corpus', 'test')
    for path in sorted((EXATHLON_PARQUET / side).glob('*.parquet'))
]).sort_values('trace_id').reset_index(drop=True)

def read_trace(trace, columns=None):
    path = exathlon_traces.set_index('trace').loc[trace, 'path']
    wanted = None if columns is None else ['t'] + [c for c in columns if c != 't']
    return pd.read_parquet(path, columns=wanted).fillna(NULL_FILL)

def iter_chunks(trace, columns=None, batch_size=8000):
    import pyarrow.parquet as pq

    path = exathlon_traces.set_index('trace').loc[trace, 'path']
    wanted = None if columns is None else ['t'] + [c for c in columns if c != 't']
    for batch in pq.ParquetFile(path).iter_batches(batch_size=int(batch_size), columns=wanted):
        yield batch.to_pandas().fillna(NULL_FILL)

def strided_rows(trace, stride, columns, batch_size=8000):
    blocks, position = [], 0
    for chunk in iter_chunks(trace, columns, batch_size):
        values = chunk[columns].to_numpy(dtype=np.float32)
        start = (-position) % stride
        if start < len(values):
            blocks.append(values[start::stride])
        position += len(values)
    return (np.vstack(blocks) if blocks
            else np.zeros((0, len(columns)), dtype=np.float32))

def load_exathlon(side, columns=None, limit=None, show_progress=False):
    chosen = exathlon_traces[exathlon_traces.side == side]
    if limit is not None:
        chosen = chosen.head(int(limit))

    rows = []
    for record in chosen.itertuples(index=False):
        frame = read_trace(record.trace, columns)
        rows.append((record.trace, frame))

    return to_canonical(rows, time_col='t')

def exathlon_truth(canonical):
    intervals = dict(zip(exathlon_labels.stream, exathlon_labels.anomalous))
    return pd.DataFrame([
        {'stream': name,
         'label': intervals_to_mask(intervals.get(name, []), length).astype(int)}
        for name, length in zip(canonical.stream,
                                canonical.Stream.map(lambda block: len(block)))])

def trace_points(side=None):
    import pyarrow.parquet as pq

    chosen = exathlon_traces if side is None else exathlon_traces[exathlon_traces.side == side]
    return int(sum(pq.read_metadata(p).num_rows for p in chosen.path))

corpus_points, test_points = trace_points('corpus'), trace_points('test')
trace_summary = pd.DataFrame([
    {'side': 'corpus', 'traces': int((exathlon_traces.side == 'corpus').sum()),
     'points': corpus_points},
    {'side': 'test', 'traces': int((exathlon_traces.side == 'test').sum()),
     'points': test_points}])
storage_summary = summarise(channels=len(VALUE_CHANNELS),
                            parquet_gb=round(exathlon_traces.bytes.sum() / 1e9, 1),
                            csv_gb=26.5)
width_costs = pd.DataFrame([
    {'selection': label, 'channels': width,
     'gb_both_sides': (corpus_points + test_points) * width * 8 / 1e9}
    for label, width in (('a bagging draw', 48), ('the UMAP arm', 24))])
display(trace_summary, storage_summary,
        width_costs.style.format({'gb_both_sides': '{:.2f}'}).hide(axis='index')
        .set_caption('full width would be {0:.0f} GB, so nothing builds one'.format(
            (corpus_points + test_points) * len(VALUE_CHANNELS) * 8 / 1e9)))

def null_census():
    import pyarrow.parquet as pq

    rows = []
    for record in exathlon_traces.itertuples(index=False):
        meta = pq.read_metadata(record.path)
        group = meta.row_group(0)
        nulls = sum(1 for i in range(group.num_columns)
                    if group.column(i).statistics is not None
                    and group.column(i).statistics.null_count > 0)
        rows.append({'side': record.side, 'trace': record.trace, 'null_columns': nulls})
    frame = pd.DataFrame(rows)
    display(frame.groupby(['side', 'null_columns']).size().rename('traces').to_frame()
            .style.set_caption('channels null in the first row group, by side'))
    return frame

exathlon_nulls = null_census()


from anomalies_scale import AUC as _auc_module
from anomalies_scale import bagging as _bagging_module
from anomalies_scale import covariance_creation as _covariance_module
from anomalies_scale import crossvalidated_thresholding as _threshold_module
from anomalies_scale import index_creation as _index_module
from anomalies_scale import interval_detection as _interval_module
from anomalies_scale import signature_computer as _signature_module
from anomalies_scale import stream_scoring as _scoring_module
from anomalies_scale import umap_projection as _umap_module
from anomalies_scale import windowing as _windowing_module
from anomalies_scale import anomaly_detection_pooled as _pooled_module
from anomalies_scale.profiling import StageRecorder

STAGE_TARGETS = [
    (_signature_module, 'compute_corpus', 'corpus signatures'),
    (_bagging_module, 'compute_corpus', 'corpus signatures'),
    (_covariance_module, 'signature_matrix', 'signature matrix'),
    (_bagging_module, 'signature_matrix', 'signature matrix'),
    (_covariance_module, 'covariance_matrix', 'covariance / whitening'),
    (_bagging_module, 'covariance_matrix', 'covariance / whitening'),
    (_pooled_module, 'build_whitening', 'covariance / whitening'),
    (_index_module, 'PooledIndex', 'index build'),
    (_pooled_module, 'PooledCorpus', 'index build'),
    (_threshold_module, 'crossvalidated_thresholds', 'calibration'),
    (_bagging_module, 'stream_folds', 'calibration'),
    (_scoring_module, 'score_streams', 'scoring'),
    (_bagging_module, 'score_streams', 'scoring'),
    (_pooled_module, 'detect_anomalies', 'detection'),
    (_auc_module, 'score_points', 'point scoring'),
    (_interval_module, 'detect_anomalous_intervals', 'per-interval detection'),
    (_umap_module, 'fit_embedding', 'UMAP fit'),
    (_umap_module, 'fit_projection', 'UMAP fit'),
    (_umap_module, 'latent_frame', 'UMAP transform'),
    (_umap_module, 'reduce_corpus', 'UMAP transform'),
    (_windowing_module, 'window_streams', 'windowing'),
]

STAGE_ROWS = []


def profiled(name, interval=0.1):
    """A `StageRecorder` with every pipeline stage patched, for one experiment.

    Used as `with profiled('experiment 1') as stages:` - the records land in `STAGE_ROWS`
    when the block exits, tagged with the experiment name.
    """
    import contextlib

    @contextlib.contextmanager
    def runner():
        recorder = StageRecorder(interval=interval)
        with recorder, recorder.instrument(STAGE_TARGETS):
            try:
                yield recorder
            finally:
                STAGE_ROWS.extend(recorder.flatten(experiment=name))

    return runner()


_ACTIVE_STAGES = {}


class _NamespaceModule:
    """Adapts a namespace dict to the getattr/setattr interface `instrument` expects.

    The notebook imports stage functions with `from x import y`, so its globals hold the
    original objects and patching the defining module never reaches them - which is why
    experiments 2 and 3 recorded nothing on the first run while experiment 1, whose work
    happens inside `bagging.one_draw`, recorded everything.
    """

    def __init__(self, namespace):
        object.__setattr__(self, '_ns', namespace)

    def __getattr__(self, key):
        return object.__getattribute__(self, '_ns').get(key)

    def __setattr__(self, key, value):
        object.__getattribute__(self, '_ns')[key] = value


NOTEBOOK_STAGE_NAMES = [
    ('compute_corpus', 'corpus signatures'),
    ('signature_matrix', 'signature matrix'),
    ('covariance_matrix', 'covariance / whitening'),
    ('build_whitening', 'covariance / whitening'),
    ('crossvalidated_thresholds', 'calibration'),
    ('score_streams', 'scoring'),
    ('score_points', 'point scoring'),
    ('detect_anomalies', 'detection'),
    ('detect_anomalous_intervals', 'per-interval detection'),
    ('fit_embedding', 'UMAP fit'),
    ('fit_projection', 'UMAP fit'),
    ('latent_frame', 'UMAP transform'),
    ('reduce_corpus', 'UMAP transform'),
    ('window_streams', 'windowing'),
    ('PooledIndex', 'index build'),
    ('PooledCorpus', 'index build'),
]


def start_stages(name, interval=0.1):
    """Begin recording stages for `name`. Paired with `stop_stages`.

    The context-manager form would mean re-indenting a hundred-line experiment cell to sit
    inside a `with`; a start/stop pair leaves the existing bodies untouched, which is worth
    more here than the exception safety a `with` would buy.
    """
    recorder = StageRecorder(interval=interval)
    here = _NamespaceModule(globals())
    targets = list(STAGE_TARGETS) + [(here, attribute, label)
                                     for attribute, label in NOTEBOOK_STAGE_NAMES]
    patch = recorder.instrument(targets)
    recorder.__enter__()
    patch.__enter__()
    _ACTIVE_STAGES[name] = (recorder, patch)
    return recorder


def stop_stages(name):
    """Stop recording, restore the patched functions, and file the records."""
    recorder, patch = _ACTIVE_STAGES.pop(name, (None, None))
    if recorder is None:
        return None
    patch.__exit__(None, None, None)
    recorder.__exit__(None, None, None)
    rows = recorder.flatten(experiment=name)
    STAGE_ROWS.extend(rows)
    if rows:
        frame = pd.DataFrame(rows)
        if STAGE_ROWS_CSV.exists():
            existing = pd.read_csv(STAGE_ROWS_CSV)
            existing = existing[existing.experiment != name]
            frame = pd.concat([existing, frame], ignore_index=True)
        frame.to_csv(STAGE_ROWS_CSV, index=False)
    return recorder


def stage_totals(rows=None):
    """One row per experiment: wall time, the largest peak seen, and the stage that set it."""
    frame = pd.DataFrame(rows if rows is not None else STAGE_ROWS)
    if frame.empty:
        return frame
    out = []
    for experiment, group in frame.groupby('experiment'):
        top = group.loc[group.peak_rss_mb.idxmax()] if group.peak_rss_mb.notna().any() else None
        out.append({
            'experiment': experiment,
            'stages_recorded': int(len(group)),
            'stage_seconds': float(group.exclusive_seconds.sum()),
            'peak_rss_mb': float(group.peak_rss_mb.max()) if top is not None else float('nan'),
            'peak_stage': None if top is None else str(top.stage),
        })
    return pd.DataFrame(out)


def stage_breakdown(rows=None):
    """Per-experiment, per-stage cost as a frame: calls, seconds, peak and delta memory."""
    frame = pd.DataFrame(rows if rows is not None else STAGE_ROWS)
    if frame.empty:
        return frame
    grouped = frame.groupby(['experiment', 'stage']).agg(
        calls=('seconds', 'size'),
        seconds=('seconds', 'sum'),
        exclusive_seconds=('exclusive_seconds', 'sum'),
        peak_rss_mb=('peak_rss_mb', 'max'),
        mean_delta_mb=('delta_rss_mb', 'mean')).reset_index()
    total = grouped.groupby('experiment').exclusive_seconds.transform('sum')
    grouped['share_of_time'] = grouped.exclusive_seconds / total.replace(0, np.nan)
    return grouped.sort_values(['experiment', 'exclusive_seconds'],
                               ascending=[True, False]).reset_index(drop=True)

EXATHLON_RESULTS = PROJECT / 'results' / 'exathlon'
EXATHLON_RESULTS.mkdir(parents=True, exist_ok=True)
STAGE_ROWS_CSV = EXATHLON_RESULTS / 'stage_rows.csv'

def save_result(name, payload):
    if isinstance(payload, pd.DataFrame):
        path = EXATHLON_RESULTS / '{0}.csv'.format(name)
        payload.to_csv(path, index=False)
    else:
        path = EXATHLON_RESULTS / '{0}.json'.format(name)
        path.write_text(json.dumps(payload, indent=1, default=float), encoding='utf-8')

    return path

def load_result(name):
    csv, js = (EXATHLON_RESULTS / '{0}.csv'.format(name),
               EXATHLON_RESULTS / '{0}.json'.format(name))
    if csv.exists():
        return pd.read_csv(csv)
    if js.exists():
        return json.loads(js.read_text(encoding='utf-8'))
    return None

### 4.1 Experiment 1 - bootstrap sampling

In [ ]:
%%capture
%%time
%%capture
import gc

from anomalies_scale.AUC import ranking_metrics
from anomalies_scale.bagging import channel_draws, one_draw
from anomalies_scale.stream_evaluation import intervals_to_mask

EXP1_DRAWS = 40
EXP1_VOTES = 1
EXP1_FEATURES = 48
OFF_MANIFOLD = 1e-3

n_channels = len(VALUE_CHANNELS)
exp1_subsets = channel_draws(n_channels, EXP1_DRAWS, EXP1_FEATURES, random_state=0)
display(summarise(draws=EXP1_DRAWS, channels_per_draw=len(exp1_subsets[0]),
                  channels_available=n_channels,
                  terms_per_draw=iisignature.siglength(
                      len(exp1_subsets[0]) + 1, EXATHLON_TRUNC),
                  truncation=EXATHLON_TRUNC))

start_stages('experiment 1')

exa_truth = None
tally, exp1_draws, exp1_started = None, [], time.perf_counter()

for number, columns in enumerate(exp1_subsets, start=1):
    drawn = [VALUE_CHANNELS[i] for i in columns]
    started = time.perf_counter()

    corpus = load_exathlon('corpus', columns=drawn)
    test = load_exathlon('test', columns=drawn)
    if exa_truth is None:
        exa_truth = exathlon_truth(test)
        lengths = {name: len(v) for name, v in zip(exa_truth.stream, exa_truth.label)}
        tally = {name: np.zeros(length, dtype=int) for name, length in lengths.items()}

    scored, info = one_draw(
        corpus, test, np.arange(len(drawn)), EXATHLON_TRUNC, GRANULARITY, VARIANCE_KEEP,
        BAND, FOLDS, STATISTIC, number, None, CONFIG['detect']['sig_tol'],
        CONFIG['detect']['tol'], NEIGHBOURS, 'mahalanobis', None, NORMALISE, RCOND,
        OFF_MANIFOLD)

    flagged = 0
    for row in scored.itertuples(index=False):
        mask = intervals_to_mask(row.anomalous, lengths[str(row.stream)])
        tally[str(row.stream)] += mask
        flagged += int(mask.sum())

    info.update(draw=number, flagged_fraction=flagged / sum(lengths.values()),
                seconds=time.perf_counter() - started)
    exp1_draws.append(info)

    del corpus, test, scored
    gc.collect()

exp1_seconds = time.perf_counter() - exp1_started
stop_stages('experiment 1')

exp1_votes = pd.DataFrame({
    'stream': list(tally),
    'votes': [tally[name] for name in tally],
})
exp1_scored = pd.DataFrame({
    'stream': list(tally),
    'Time': [np.arange(lengths[name], dtype=float) for name in tally],
    'Stream': [np.zeros((lengths[name], 1)) for name in tally],
    'anomalous': [[[int(lo), int(hi) - 1] for lo, hi in segments(tally[name] >= EXP1_VOTES)]
                  for name in tally],
})

exp1 = evaluate_predictions(exp1_scored, truth=exa_truth, show_progress=True)
exp1['throughput'] = {
    'seconds': exp1_seconds, 'draws': EXP1_DRAWS,
    'channels_per_draw': int(len(exp1_subsets[0])),
    'points_scored': int(exp1['n_points']),
    'points_per_second': exp1['n_points'] / exp1_seconds,
    'seconds_per_draw': exp1_seconds / EXP1_DRAWS,
    'peak_rss_mb': peak_memory_bytes() / 1e6,
}

exp1_curve = pd.DataFrame([
    {'votes': v,
     **{k: prf(np.concatenate([tally[n] >= v for n in tally]),
               np.concatenate([np.asarray(exa_truth.set_index('stream').label[n], dtype=bool)
                               for n in tally]))[k]
        for k in ('precision', 'recall', 'f1')}}
    for v in range(1, EXP1_DRAWS + 1)])
display(pd.DataFrame(exp1_draws).style.format(precision=3).hide(axis='index')
        .set_caption('per draw'),
        summarise(**{k: round(v, 2) if isinstance(v, float) else v
                     for k, v in exp1['throughput'].items()}),
        exp1_curve.style.format(precision=3).hide(axis='index')
        .set_caption('vote curve'))

by_stream = exa_truth.set_index('stream').label
exp1['ranking'] = ranking_metrics(
    np.concatenate([tally[name] / float(EXP1_DRAWS) for name in tally]),
    np.concatenate([np.asarray(by_stream[name], dtype=bool) for name in tally]))
if exp1['ranking']:
    display(summarise(**exp1['ranking']).style.set_caption(
        'ranking, vote fraction over {0} draws'.format(EXP1_DRAWS)))

save_result('experiment1', exp1)
save_result('experiment1_draws', pd.DataFrame(exp1_draws))
save_result('experiment1_votes', exp1_curve)

### 4.2 Window sampling with jittered offsets
Instead of detecting anomalies within the series, we use the same methods to simply detect anomalous windows. There is no reduction of the intervals to classify if section are normal or not.

N.B. This is what's performed above across the dyadic intervals to reduce the size, but here for simplicity we look at windows of a fixed width for the corpus.

In [ ]:
%%capture
WINDOW_SIZES = (64, 128, 256)
MAJORITY = 2.0 / 3.0
JITTER_SEED = 0

TEST_WINDOWS_PER_TRACE = 500
SCORE_BATCH = 500

def corpus_windows_for(size, streams):
    return window_streams(streams, size, min_points=size + 1)

def window_budget(streams=None, test_streams=None, sizes=WINDOW_SIZES,
                  terms=None):
    terms = terms if terms is not None else iisignature.siglength(UMAP_DIMENSION + 1,
                                                                  WINDOW_TRUNC)
    points = int(sum(len(np.asarray(v)) for v in streams.Stream))
    test = len(test_streams) * TEST_WINDOWS_PER_TRACE

    budget = pd.DataFrame([
        {'width': size, 'corpus_windows': points // size, 'test_windows': test,
         'signatures_mb': (points // size) * terms * 8 / 1e6,
         'svd_factor_mb': min(points // size, terms) * terms * 8 / 1e6}
        for size in sizes])
    display(summarise(corpus_points=points, traces=len(streams), terms=terms,
                      width=UMAP_DIMENSION + 1, truncation=WINDOW_TRUNC,
                      score_batch_mb=round(SCORE_BATCH * terms * 8 / 1e6),
                      unbatched_test_mb=round(test * terms * 8 / 1e6)),
            budget.style.format({'corpus_windows': '{:,}', 'test_windows': '{:,}',
                                 'signatures_mb': '{:,.0f}',
                                 'svd_factor_mb': '{:,.0f}'}).hide(axis='index'))

def jittered_windows(canonical, truth, size, per_trace, seed=JITTER_SEED):
    rng = np.random.default_rng(seed)
    labels = dict(zip(truth.stream, truth.label))

    rows, flags, provenance = [], [], []
    for record in canonical.itertuples(index=False):
        values = np.asarray(record.Stream, dtype=float)
        time_axis = np.asarray(record.Time, dtype=float)
        mask = np.asarray(labels[record.stream], dtype=bool)
        if len(values) <= size:
            continue

        starts = rng.integers(0, len(values) - size, size=int(per_trace))
        for number, start in enumerate(sorted(starts)):
            stop = start + size
            rows.append({'stream': '{0}__w{1:05d}'.format(record.stream, number),
                         'Time': time_axis[start:stop],
                         'Stream': values[start:stop]})
            flags.append(mask[start:stop].mean() >= MAJORITY)
            provenance.append((record.stream, int(start)))

    windows = pd.DataFrame(rows)
    return windows, np.asarray(flags), provenance

def window_report(name, predicted, actual, seconds, extra=None, scores=None):
    from anomalies_scale.AUC import ranking_metrics
    from anomalies_scale.stream_evaluation import prf

    record = dict(prf(np.asarray(predicted, dtype=bool), np.asarray(actual, dtype=bool)))
    if scores is not None:
        scores = np.asarray(scores, dtype=float)
        record.update(ranking_metrics(scores, np.asarray(actual, dtype=bool)))
        finite = scores[np.isfinite(scores)]
        if finite.size:
            record.update({
                'score_median': float(np.median(finite)),
                'score_p05': float(np.percentile(finite, 5)),
                'score_p95': float(np.percentile(finite, 95)),
            })
    record.update({
        'experiment': name,
        'n_windows': int(len(actual)),
        'positive_rate': float(np.mean(actual)),
        'flagged_rate': float(np.mean(predicted)),
        'seconds': seconds,
        'windows_per_second': len(actual) / seconds if seconds else float('nan'),
        'peak_rss_mb': peak_memory_bytes() / 1e6,
    })
    record.update(extra or {})
    return record

UMAP_FIT_STRIDE = 250
WINDOW_GRANULARITY = 0
METRIC_FIT_WINDOWS = 12000

def window_detector(corpus_windows, test_windows, trunc=WINDOW_TRUNC,
                    fit_windows=METRIC_FIT_WINDOWS, variance_keep=None,
                    rcond=None):
    timings = {}

    started = time.perf_counter()
    sample = (corpus_windows.sample(int(fit_windows), random_state=0)
              if len(corpus_windows) > fit_windows else corpus_windows)
    fit_corpus = compute_corpus(sample, trunc=trunc, granularity=WINDOW_GRANULARITY)

    width = as_matrix(corpus_windows.Stream.iloc[0]).shape[1] + 1
    normaliser = fit_normaliser(fit_corpus, method=NORMALISE, width=width)
    fit_corpus = normaliser.transform(fit_corpus)
    signatures, _ = signature_matrix(fit_corpus)

    values, right, n_rows = spectrum(signatures)
    rank = retained_rank(values,
                         VARIANCE_KEEP if variance_keep is None else variance_keep,
                         RCOND if rcond is None else rcond)
    metric = Whitening(right[:rank], np.sqrt(n_rows) / values[:rank])
    timings['fit_metric'] = time.perf_counter() - started
    timings['fit_windows'] = len(sample)
    timings['rank'] = int(rank)
    timings['terms'] = int(signatures.shape[1])
    del signatures, values, right
    gc.collect()

    started = time.perf_counter()
    thresholds, calibration = crossvalidated_thresholds(
        fit_corpus, metric, k=FOLDS, statistic=STATISTIC, band=BAND, neighbours=NEIGHBOURS,
        windowed=True)
    threshold = float(np.mean(list(thresholds.values())))
    timings.update({
        'corpus_median_distance': float(calibration['median_distance'].mean()),
        'corpus_p90_distance': float(calibration['p90_distance'].mean()),
        'corpus_p99_distance': float(calibration['p99_distance'].mean()),
        'corpus_max_distance': float(calibration['max_distance'].max()),
        'n_calibration': int(calibration['n_calibration'].sum()),
    })
    timings['calibrate'] = time.perf_counter() - started
    del fit_corpus
    gc.collect()

    started = time.perf_counter()
    reference = []
    for first in range(0, len(corpus_windows), SCORE_BATCH):
        batch = corpus_windows.iloc[first:first + SCORE_BATCH]
        frame = normaliser.transform(
            compute_corpus(batch, trunc=trunc, granularity=WINDOW_GRANULARITY))
        batch_signatures, _ = signature_matrix(frame)
        reference.append(np.ascontiguousarray(metric.apply(batch_signatures), dtype='float32'))
        del frame, batch_signatures
    reference = np.vstack(reference)
    index = faiss.IndexFlatL2(reference.shape[1])
    index.add(reference)
    timings['sign_corpus'] = time.perf_counter() - started
    timings['reference_vectors'] = len(reference)

    started = time.perf_counter()
    scores = []
    for first in range(0, len(test_windows), SCORE_BATCH):
        batch = test_windows.iloc[first:first + SCORE_BATCH]
        frame = normaliser.transform(
            compute_corpus(batch, trunc=trunc, granularity=WINDOW_GRANULARITY))
        batch_signatures, _ = signature_matrix(frame)
        queries = np.ascontiguousarray(metric.apply(batch_signatures), dtype='float32')
        distances, _ = index.search(queries, NEIGHBOURS)
        scores.append(distances[:, NEIGHBOURS - 1])
        del frame, batch_signatures, queries, distances
    timings['score'] = time.perf_counter() - started

    score = np.concatenate(scores)
    timings['windows_per_second'] = len(score) / max(timings['score'], 1e-9)
    return score > threshold, score, threshold, timings


### 4.3 Experiment 2 - a UMAP whole-window detector

As with the bagging method, the dataset is simply too large to load so an alternative dimensionality reduction is UMAP. Here we apply it to the windowed streams, this provides context to them allowing for the UMAP to be valid.

In [ ]:
%%capture
import gc

from anomalies_scale.umap_projection import embedding_quality, fit_embedding
from anomalies_scale.covariance_creation import (
    Whitening, retained_rank, signature_matrix, spectrum)
from anomalies_scale.crossvalidated_thresholding import crossvalidated_thresholds

start_stages('experiment 2')


started = time.perf_counter()
ambient = np.vstack([
    strided_rows(name, UMAP_FIT_STRIDE, VALUE_CHANNELS)
    for name in exathlon_traces[exathlon_traces.side == 'corpus'].trace]).astype(np.float64)

projection, umap_info = fit_embedding(ambient, UMAP_DIMENSION, context=1)
umap_fit_seconds = time.perf_counter() - started

UMAP_DIAGNOSTICS = {
    'latent_channels': UMAP_DIMENSION,
    'ambient_columns': umap_info['ambient_columns'],
    'n_fit_rows': umap_info['n_fit_rows'],
    'fit_stride': UMAP_FIT_STRIDE,
    'n_neighbors': projection.params.get('n_neighbors'),
    'min_dist': projection.params.get('min_dist'),
    'knn_neighbours': projection.params.get('knn_neighbours'),
    'fit_seconds': umap_fit_seconds,
    'trustworthiness': embedding_quality(
        umap_info['fit_ambient'], umap_info['embedding'],
        n_neighbours=projection.params.get('n_neighbors', 15)),
    'knn_relative_error': umap_info['knn_relative_error'],
    'peak_rss_mb': peak_memory_bytes() / 1e6,
}
display(summarise(**UMAP_DIAGNOSTICS))

del ambient, umap_info
gc.collect()

def latent_side(side, batch_size=8000):
    rows = []
    for name in exathlon_traces[exathlon_traces.side == side].trace:
        clocks, pieces = [], []
        for chunk in iter_chunks(name, VALUE_CHANNELS, batch_size):
            clocks.append(chunk['t'].to_numpy())
            pieces.append(projection.transform(
                chunk[VALUE_CHANNELS].to_numpy(dtype=np.float32)))
        latent = np.vstack(pieces)
        block = pd.DataFrame(latent, columns=['c{0}'.format(i) for i in range(latent.shape[1])])
        block.insert(0, 't', np.concatenate(clocks))
        rows.append((name, block))
        del clocks, pieces, latent
        gc.collect()
    return to_canonical(rows, time_col='t')

started = time.perf_counter()
latent_corpus = latent_side('corpus')
latent_test = latent_side('test')
project_seconds = time.perf_counter() - started
display(summarise(
    project_seconds=round(project_seconds),
    corpus_gb=round(sum(v.nbytes for v in latent_corpus.Stream) / 1e9, 2),
    test_gb=round(sum(v.nbytes for v in latent_test.Stream) / 1e9, 2)))

latent_truth = exathlon_truth(latent_test)
window_budget(streams=latent_corpus, test_streams=latent_test)

exp2_rows = []
for size in WINDOW_SIZES:
    corpus_windows = corpus_windows_for(size, latent_corpus)
    test_windows, actual, _ = jittered_windows(
        latent_test, latent_truth, size, TEST_WINDOWS_PER_TRACE)

    started = time.perf_counter()
    flags, scores, threshold, timings = window_detector(corpus_windows, test_windows)
    seconds = time.perf_counter() - started

    exp2_rows.append(window_report(
        'umap window W={0}'.format(size), flags, actual, seconds, scores=scores,
        extra={'window': size, 'threshold': threshold, 'corpus_windows': len(corpus_windows),
               'latent_channels': UMAP_DIMENSION, 'trunc': WINDOW_TRUNC,
               'umap_fit_seconds': umap_fit_seconds, 'project_seconds': project_seconds,
               **timings}))
    del corpus_windows, test_windows
    gc.collect()

stop_stages('experiment 2')
exp2 = pd.DataFrame(exp2_rows)
display(exp2.style.format(precision=4).hide(axis='index'))

save_result('experiment2', exp2)
save_result('umap_diagnostics', UMAP_DIAGNOSTICS)

### 4.4 Experiment 3 - the same detector on the bootstrap sample

In [ ]:
%%capture
%%time
from anomalies_scale.bagging import channel_draws

start_stages('experiment 3')

EXP3_DRAWS = 12
EXP3_VOTES = 1
EXP3_FEATURES = 63

exp3_subsets = channel_draws(len(VALUE_CHANNELS), EXP3_DRAWS, EXP3_FEATURES, random_state=0)
exp3_tally, exp3_actual, exp3_windows = {}, {}, {}
exp3_score = {}
exp3_started = time.perf_counter()

for number, columns in enumerate(exp3_subsets, start=1):
    drawn = [VALUE_CHANNELS[i] for i in columns]
    corpus = load_exathlon('corpus', columns=drawn)
    test = load_exathlon('test', columns=drawn)
    truth = exathlon_truth(test)

    for size in WINDOW_SIZES:
        corpus_windows = corpus_windows_for(size, corpus)
        test_windows, actual, _ = jittered_windows(test, truth, size, TEST_WINDOWS_PER_TRACE)
        flags, scores, threshold, timings = window_detector(corpus_windows, test_windows)

        if size not in exp3_tally:
            exp3_tally[size] = np.zeros(len(actual), dtype=int)
            exp3_score[size] = np.zeros(len(actual), dtype=float)
            exp3_actual[size] = actual
            exp3_windows[size] = len(corpus_windows)
        exp3_tally[size] += flags.astype(int)
        exp3_score[size] += np.asarray(scores, dtype=float) / max(float(threshold), 1e-12)
        del corpus_windows, test_windows
        gc.collect()

    del corpus, test
    gc.collect()

exp3_seconds = time.perf_counter() - exp3_started
stop_stages('experiment 3')

exp3_rows = []
for size in WINDOW_SIZES:
    exp3_rows.append(window_report(
        'bagged window W={0}'.format(size), exp3_tally[size] >= EXP3_VOTES, exp3_actual[size],
        exp3_seconds / len(WINDOW_SIZES), scores=exp3_score[size] / EXP3_DRAWS,
        extra={'window': size, 'draws': EXP3_DRAWS, 'votes': EXP3_VOTES,
               'channels_per_draw': EXP3_FEATURES, 'trunc': WINDOW_TRUNC,
               'corpus_windows': exp3_windows[size]}))

exp3 = pd.DataFrame(exp3_rows)
display(exp3.style.format(precision=4).hide(axis='index'))

save_result('experiment3', exp3)

### 4.5 The TimeEval benchmark protocol

In [ ]:
%%capture
%%time
import gc

from anomalies_scale.AUC import POINT_SCORE_COLUMN, ranking_metrics, score_points
from anomalies_scale.covariance_creation import covariance_matrix, signature_matrix
from anomalies_scale.index_creation import PooledIndex
from anomalies_scale.windowing import window_streams

TIMEEVAL = EXATHLON / 'timeeval' / 'multivariate' / 'Exathlon'

TE_WINDOW = 64
TE_TRUNC = 2
TE_SEGMENTS = 6
TE_FOLDS = 5
TE_QUERY_STRIDE = 4
TE_RCOND = 1e-2

def timeeval_datasets():
    rows = []
    for path in sorted(TIMEEVAL.glob('*.test.csv')):
        stem = path.name[:-len('.test.csv')]
        train = TIMEEVAL / (stem + '.train.csv')
        if not train.exists():
            continue
        if pd.read_csv(train, usecols=['is_anomaly']).is_anomaly.mean() > 0:
            continue
        rows.append(stem)
    return rows

def timeeval_load(stem, part):
    frame = pd.read_csv(TIMEEVAL / '{0}.{1}.csv'.format(stem, part))
    return (frame.drop(columns=['timestamp', 'is_anomaly']),
            frame['is_anomaly'].to_numpy(dtype=int))

def timeeval_streams(values, name, segments=1):
    rows, edges = [], np.linspace(0, len(values), segments + 1, dtype=int)
    for i in range(segments):
        piece = values.iloc[edges[i]:edges[i + 1]].reset_index(drop=True).copy()
        piece.insert(0, 't', np.arange(len(piece), dtype=float))
        rows.append(('{0}seg{1:02d}'.format(name, i), piece))
    return to_canonical(rows, time_col='t')

TE_EXCLUDE = ('5_1_100000_64',)
te_names = [s for s in timeeval_datasets()
            if not s.startswith(TE_EXCLUDE)]
te_survey = []
for stem in te_names:
    train, _ = timeeval_load(stem, 'train')
    test, labels = timeeval_load(stem, 'test')
    te_survey.append({'dataset': stem, 'channels': train.shape[1], 'train_rows': len(train),
                      'test_rows': len(test), 'anomaly_rate': float(labels.mean()),
                      'trace': stem.rsplit('-', 1)[0]})
te_survey = pd.DataFrame(te_survey)

display(summarise(datasets=len(te_survey), traces=te_survey.trace.nunique(),
                  channels_min=te_survey.channels.min(),
                  channels_median=te_survey.channels.median(),
                  channels_mean=round(te_survey.channels.mean(), 1),
                  channels_max=te_survey.channels.max(),
                  train_rows=int(te_survey.train_rows.sum()),
                  test_rows=int(te_survey.test_rows.sum()),
                  anomaly_rate=round(te_survey.anomaly_rate.mean(), 4)),
        te_survey.style.format({'anomaly_rate': '{:.4f}', 'train_rows': '{:,}',
                                'test_rows': '{:,}'}).hide(axis='index'))

TE_JITTER_SIZES = WINDOW_SIZES
TE_JITTER_PER_DATASET = 10000
TE_JITTER_LIMIT = None
TE_UMAP_MAX = 12
TE_BAG_DRAWS = 8
TE_BAG_FRACTION = 0.6
TE_BAG_MIN_CHANNELS = 6
def timeeval_canonical(stem, transform=None, columns=None):
    """Train and test halves as canonical streams, plus the point labels.

    `columns` selects a channel subset (bagging); `transform` maps the value matrix through a
    fitted projection (UMAP). Both are applied identically to the two halves, which is the
    point - a projection fitted on train and not applied to test would compare unlike things.
    """
    train, _ = timeeval_load(stem, 'train')
    test, labels = timeeval_load(stem, 'test')
    if columns is not None:
        train, test = train.iloc[:, columns], test.iloc[:, columns]
    if transform is not None:
        train = pd.DataFrame(transform(train.to_numpy(dtype=np.float32)))
        test = pd.DataFrame(transform(test.to_numpy(dtype=np.float32)))
        train.columns = ['c{0}'.format(i) for i in range(train.shape[1])]
        test.columns = list(train.columns)

    corpus = timeeval_streams(train, '{0}_train'.format(stem), segments=TE_SEGMENTS)
    evaluation = timeeval_streams(test, '{0}_test'.format(stem), segments=1)
    truth = pd.DataFrame([{'stream': evaluation.stream.iloc[0],
                           'label': np.asarray(labels, dtype=int)}])
    return corpus, evaluation, truth, np.asarray(labels, dtype=bool)
def paint_windows(flags, provenance, size, length):
    """Flagged windows back onto the series: the predicted mask and what was covered at all."""
    predicted = np.zeros(length, dtype=bool)
    covered = np.zeros(length, dtype=bool)
    for flag, (_, start) in zip(np.asarray(flags, dtype=bool), provenance):
        stop = min(start + size, length)
        covered[start:stop] = True
        if flag:
            predicted[start:stop] = True
    return predicted, covered
def point_level(predicted, covered, truth):
    """Point metrics over the covered part of the series, plain and point-adjusted."""
    if not covered.any():
        return {}
    seen_truth, seen_predicted = truth[covered], predicted[covered]
    record = {'point_{0}'.format(k): v for k, v in prf(seen_predicted, seen_truth).items()}
    adjusted = predicted.copy()
    for lo, hi in segments(truth):
        if adjusted[lo:hi].any():
            adjusted[lo:hi] = True
    record.update({'adj_{0}'.format(k): v for k, v in prf(adjusted[covered], seen_truth).items()})
    record['covered_fraction'] = float(covered.mean())
    return record
def _run_windows(corpus, evaluation, truth_frame, size):
    """Tile the corpus, draw jittered test windows, score them whole."""
    corpus_windows = corpus_windows_for(size, corpus)
    test_windows, actual, provenance = jittered_windows(
        evaluation, truth_frame, size, TE_JITTER_PER_DATASET)
    if not len(test_windows) or not len(corpus_windows):
        return None
    flags, scores, threshold, timings = window_detector(corpus_windows, test_windows)
    timings['corpus_windows'] = len(corpus_windows)
    return flags, np.asarray(scores, dtype=float), actual, provenance, threshold, timings


#### Building detectors

In [ ]:
%%capture
%%time
def timeeval_detector(stem, rcond=TE_RCOND, window=TE_WINDOW):
    started = time.perf_counter()
    train, _ = timeeval_load(stem, 'train')
    test, labels = timeeval_load(stem, 'test')
    truth = labels.astype(bool)

    corpus_windows = window_streams(timeeval_streams(train, 'tr', TE_SEGMENTS), window,
                                    min_points=window + 1)
    corpus = compute_corpus(corpus_windows, trunc=TE_TRUNC, granularity=0)
    normaliser = fit_normaliser(corpus, method=NORMALISE, width=train.shape[1] + 1)
    corpus = normaliser.transform(corpus)
    signatures, terms = signature_matrix(corpus)

    _, info = covariance_matrix(signatures, variance_keep=VARIANCE_KEEP, form='inv_sqrt',
                                rcond=rcond)
    metric = info['whitening']
    whitened = np.ascontiguousarray(metric.apply(signatures), dtype='float32')
    index = faiss.IndexFlatL2(whitened.shape[1])
    index.add(whitened)
    depths = corpus['depth'].to_numpy(dtype=int)
    engine = PooledIndex(index, depth=depths, split=np.zeros(len(depths), dtype=int),
                         band=BAND, terms=terms)
    fitted = time.perf_counter() - started

    thresholds, calibration = crossvalidated_thresholds(
        corpus, metric, k=TE_FOLDS, statistic=STATISTIC, band=BAND, neighbours=NEIGHBOURS,
        windowed=True)

    scoring = time.perf_counter()
    scored = score_points(timeeval_streams(test, 'te', 1), engine, metric, thresholds,
                          window=window, stride=TE_QUERY_STRIDE, span=window,
                          normaliser=normaliser, trunc=TE_TRUNC, neighbours=NEIGHBOURS)
    scores = np.concatenate(list(scored[POINT_SCORE_COLUMN]))[:len(truth)]
    actual = truth[:len(scores)]
    scoring = time.perf_counter() - scoring

    record = {'dataset': stem, 'trace': stem.rsplit('-', 1)[0], 'channels': train.shape[1],
              'terms': int(signatures.shape[1]), 'rank': int(info['rank']),
              'corpus_windows': len(corpus_windows),
              'rows_per_term': len(corpus_windows) / float(signatures.shape[1]),
              'anomaly_rate': float(actual.mean()), 'flagged': float((scores > 1.0).mean()),
              'corpus_p99_distance': float(calibration['p99_distance'].mean()),
              'score_median': float(np.median(scores)),
              'fit_seconds': fitted, 'score_seconds': scoring,
              'points_per_second': len(scores) / max(scoring, 1e-9),
              'seconds': time.perf_counter() - started,
              'peak_rss_mb': peak_memory_bytes() / 1e6}
    record.update(prf(scores > 1.0, actual))
    record.update(ranking_metrics(scores, actual))
    return record

te_rows, te_started = [], time.perf_counter()
for number, stem in enumerate(te_names, start=1):
    te_rows.append(timeeval_detector(stem))
    row = te_rows[-1]
    gc.collect()

timeeval = pd.DataFrame(te_rows)
te_seconds = time.perf_counter() - te_started

by_trace = timeeval.groupby('trace').agg(
    datasets=('dataset', 'size'), channels=('channels', 'first'),
    anomaly_rate=('anomaly_rate', 'first'), roc_auc=('roc_auc', 'mean'),
    pr_auc_lift=('pr_auc_lift', 'mean'))

headline = summarise(
    roc_auc_over_datasets=round(timeeval.roc_auc.mean(), 4),
    roc_auc_over_traces=round(by_trace.roc_auc.mean(), 4),
    roc_auc_median=round(timeeval.roc_auc.median(), 4),
    pr_auc_lift=round(timeeval.pr_auc_lift.mean(), 2),
    above_0_9=int((timeeval.roc_auc > 0.9).sum()),
    below_0_5=int((timeeval.roc_auc < 0.5).sum()),
    datasets=len(timeeval),
    points_per_second=round(timeeval.points_per_second.mean()),
    total_seconds=round(te_seconds),
    peak_rss_mb=round(timeeval.peak_rss_mb.max()),
    flagged=round(timeeval.flagged.mean(), 3))

display(timeeval[['dataset', 'channels', 'terms', 'rank', 'corpus_windows', 'anomaly_rate',
                  'roc_auc', 'pr_auc', 'pr_auc_lift', 'flagged', 'precision', 'recall', 'f1']]
        .sort_values('roc_auc', ascending=False)
        .style.format(precision=4).hide(axis='index').set_caption('per dataset'),
        by_trace.style.format(precision=4).set_caption('by distinct test trace'),
        headline.style.set_caption('headline'))

save_result('timeeval_exathlon', timeeval)

#### AUC calculations

In [ ]:
%%capture
%%time
sweep = pd.read_csv(EXATHLON_RESULTS / 'timeeval' / 'rank_sweep.csv')

by_rcond = sweep.groupby('rcond').agg(roc_auc=('roc_auc', 'mean'),
                                      pr_auc_lift=('pr_auc_lift', 'mean'),
                                      mean_rank=('rank', 'mean'))
display(by_rcond.style.format(precision=4)
        .set_caption('mean ROC-AUC at each fixed rank rule, over all 22 datasets'))

probe = {'5_1_100000_64-33', '2_1_100000_60-20'}
held = sweep[~sweep.dataset.isin(probe)].groupby('rcond').roc_auc.mean()
display(held.rename('roc_auc').to_frame().style.format(precision=4)
        .set_caption('the same, on the 20 datasets not used to choose TE_RCOND '
                     '(best {0:g} at {1:.4f}; TE_RCOND {2:.4f})'.format(
                         held.idxmax(), held.max(), held.get(TE_RCOND, float('nan')))))

oracle = sweep.loc[sweep.groupby('dataset').roc_auc.idxmax()]
display(summarise(
    oracle_roc_auc=round(oracle.roc_auc.mean(), 4),
    fixed_rule_roc_auc=round(by_rcond.roc_auc.max(), 4),
    gap=round(oracle.roc_auc.mean() - by_rcond.roc_auc.max(), 4),
    rcond_chosen=', '.join('{0:g} x{1}'.format(r, c)
                           for r, c in oracle.rcond.value_counts().items()))
    .style.set_caption('per-dataset oracle - an upper bound, it selects on the '
                       'reported metric'))

spread = timeeval.groupby('trace').roc_auc.agg(['min', 'max', 'size'])
spread['spread'] = spread['max'] - spread['min']
display(spread.sort_values('spread', ascending=False).style.format(precision=4)
        .set_caption('within-trace spread across training segments'))

#### The jittered-window detector

In [ ]:
%%capture
%%time


def detector_plain(stem, size, channels):
    """Pooled windows over the raw channels."""
    corpus, evaluation, truth_frame, truth = timeeval_canonical(stem)
    out = _run_windows(corpus, evaluation, truth_frame, size)
    return (None, truth) if out is None else (out, truth)


def detector_umap(stem, size, channels):
    """Pooled windows over a UMAP reduction, with the latent size scaled to the dataset."""
    from anomalies_scale.umap_projection import fit_embedding

    latent = min(TE_UMAP_MAX, max(2, channels // 2))
    if latent >= channels:
        return 'skipped: {0} channels, nothing to reduce'.format(channels), None
    train, _ = timeeval_load(stem, 'train')
    projection, info = fit_embedding(train.to_numpy(dtype=np.float32), latent, context=1)
    corpus, evaluation, truth_frame, truth = timeeval_canonical(
        stem, transform=projection.transform)
    out = _run_windows(corpus, evaluation, truth_frame, size)
    if out is None:
        return None, truth
    out[5]['latent_channels'] = latent
    out[5]['trustworthiness'] = float(info.get('trustworthiness', float('nan')))
    return out, truth


def detector_bagged(stem, size, channels):
    """Pooled windows over random channel subsets; scores averaged, flags unioned."""
    from anomalies_scale.bagging import channel_draws

    if channels < TE_BAG_MIN_CHANNELS:
        return 'skipped: {0} channels, too narrow to bag'.format(channels), None
    features = max(2, int(round(channels * TE_BAG_FRACTION)))
    subsets = channel_draws(channels, TE_BAG_DRAWS, features, random_state=0)

    total, votes, actual, provenance, thresholds, timings = None, None, None, None, [], {}
    for columns in subsets:
        corpus, evaluation, truth_frame, truth = timeeval_canonical(
            stem, columns=list(columns))
        out = _run_windows(corpus, evaluation, truth_frame, size)
        if out is None:
            continue
        flags, scores, actual, provenance, threshold, timings = out
        normalised = scores / max(float(threshold), 1e-12)
        total = normalised if total is None else total + normalised
        votes = flags.astype(int) if votes is None else votes + flags.astype(int)
        thresholds.append(threshold)
        del corpus, evaluation
        gc.collect()

    if total is None:
        return None, None
    timings.update(draws=len(thresholds), channels_per_draw=features)
    return ((votes >= 1), total / len(thresholds), actual, provenance,
            1.0, timings), truth


TE_DETECTORS = [('plain', detector_plain)]

te_jitter_rows, te_skipped = [], []
te_jitter_names = te_names if TE_JITTER_LIMIT is None else te_names[:TE_JITTER_LIMIT]
start_stages('timeeval jitter')
te_jitter_started = time.perf_counter()

for number, stem in enumerate(te_jitter_names, start=1):
    channels = int(timeeval_load(stem, 'train')[0].shape[1])
    for label, detector in TE_DETECTORS:
        for size in TE_JITTER_SIZES:
            started = time.perf_counter()
            try:
                out, truth = detector(stem, size, channels)
            except Exception as error:
                te_skipped.append({'dataset': stem, 'detector': label, 'window': size,
                                   'reason': '{0}: {1}'.format(type(error).__name__, error)})
                continue
            if isinstance(out, str):
                te_skipped.append({'dataset': stem, 'detector': label, 'window': size,
                                   'reason': out})
                continue
            if out is None:
                te_skipped.append({'dataset': stem, 'detector': label, 'window': size,
                                   'reason': 'no windows at this width'})
                continue

            flags, scores, actual, provenance, threshold, timings = out
            seconds = time.perf_counter() - started
            predicted, covered = paint_windows(flags, provenance, size, len(truth))
            te_jitter_rows.append(window_report(
                'timeeval {0} W={1}'.format(label, size), flags, actual, seconds,
                scores=scores,
                extra={'dataset': stem, 'detector': label, 'window': size,
                       'threshold': threshold, 'channels': channels,
                       'trunc': WINDOW_TRUNC,
                       **point_level(predicted, covered, truth), **timings}))
            gc.collect()
    print('{0:>2}/{1} {2} ({3} channels)'.format(
        number, len(te_jitter_names), stem, channels), flush=True)

te_jitter_seconds = time.perf_counter() - te_jitter_started
stop_stages('timeeval jitter')

te_jitter = pd.DataFrame(te_jitter_rows)
save_result('timeeval_jitter', te_jitter)
if te_skipped:
    save_result('timeeval_jitter_skipped', pd.DataFrame(te_skipped))

REPORT = ['roc_auc', 'pr_auc', 'pr_auc_lift', 'precision', 'recall', 'f1', 'point_f1',
          'adj_f1', 'positive_rate', 'flagged_rate', 'covered_fraction',
          'windows_per_second', 'peak_rss_mb']
present = [c for c in REPORT if c in te_jitter.columns]

grid = te_jitter.groupby(['detector', 'window'])[present].mean()
display(grid.style.format(precision=4).set_caption(
    'TimeEval jittered windows: {0} detector(s) x {1} width(s), mean over {2} dataset(s); '
    '{3:.0f}s'.format(te_jitter.detector.nunique(), te_jitter.window.nunique(),
                      te_jitter.dataset.nunique(), te_jitter_seconds)))

display(te_jitter.groupby('detector').agg(
    runs=('roc_auc', 'size'), datasets=('dataset', 'nunique'),
    mean_auc=('roc_auc', 'mean'), best_auc=('roc_auc', 'max'),
    inverted=('roc_auc', lambda s: int((s < 0.35).sum())),
    strong=('roc_auc', lambda s: int((s > 0.75).sum())),
    seconds=('seconds', 'sum')).style.format(precision=3).set_caption(
        'per detector, over all widths and datasets'))

if te_skipped:
    display(pd.DataFrame(te_skipped).groupby(['detector', 'reason']).size()
            .rename('runs').to_frame().style.set_caption('skipped combinations'))


#### Comparing detected anomalies to labels

In [ ]:
%%time
MARK_WIDTH = 256
MARK_COUNT = None

_summary = load_result('timeeval_jitter')
MARK_DETECTOR = 'plain'
_at_width = _summary[_summary.window == MARK_WIDTH]
if 'detector' in _at_width.columns:
    _at_width = _at_width[_at_width.detector == MARK_DETECTOR]
_at_width = _at_width.sort_values('roc_auc', ascending=False)
MARK_DATASETS = (list(_at_width.dataset) if MARK_COUNT is None
                 else list(_at_width.dataset.head(MARK_COUNT))
                 + list(_at_width.dataset.tail(MARK_COUNT)))
_auc_of = dict(zip(_at_width.dataset, _at_width.roc_auc))


def paint_scores(scores, provenance, size, length):
    """Window scores back onto the series: the largest score covering each point."""
    painted = np.full(length, np.nan)
    for score, (_, start) in zip(np.asarray(scores, dtype=float), provenance):
        stop = min(start + size, length)
        block = painted[start:stop]
        painted[start:stop] = np.where(np.isnan(block), score, np.maximum(block, score))
    return painted


def window_youden_cut(scores, actual):
    """The ROC point maximising TPR - FPR over the sampled windows."""
    from sklearn.metrics import roc_curve

    actual = np.asarray(actual, dtype=bool)
    scores = np.where(np.isfinite(scores), np.asarray(scores, dtype=float), 0.0)
    if actual.all() or not actual.any() or np.ptp(scores) == 0:
        return float('nan'), np.zeros(scores.shape, dtype=bool)
    false_rate, true_rate, cuts = roc_curve(actual, scores)
    cut = float(cuts[int(np.argmax(true_rate - false_rate))])
    return cut, scores >= cut


def window_f1_cut(scores, actual):
    """The cut maximising F1 - the same scores under a criterion that charges for precision."""
    from sklearn.metrics import precision_recall_curve

    actual = np.asarray(actual, dtype=bool)
    scores = np.where(np.isfinite(scores), np.asarray(scores, dtype=float), 0.0)
    if actual.all() or not actual.any() or np.ptp(scores) == 0:
        return float('nan'), np.zeros(scores.shape, dtype=bool)
    precision, recall, cuts = precision_recall_curve(actual, scores)
    harmonic = 2 * precision[:-1] * recall[:-1] / np.maximum(
        precision[:-1] + recall[:-1], 1e-12)
    if not len(harmonic):
        return float('nan'), np.zeros(scores.shape, dtype=bool)
    cut = float(cuts[int(np.nanargmax(harmonic))])
    return cut, scores >= cut


def plot_timeeval_marks(stem, size=MARK_WIDTH):
    """One dataset: the series, the painted score, and the flags against the truth."""
    corpus, evaluation, truth_frame, truth = timeeval_canonical(stem)
    corpus_windows = corpus_windows_for(size, corpus)
    test_windows, actual, provenance = jittered_windows(
        evaluation, truth_frame, size, TE_JITTER_PER_DATASET)
    flags, scores, threshold, _ = window_detector(corpus_windows, test_windows)

    youden, youden_flags = window_youden_cut(scores, actual)
    best_cut, best_flags = window_f1_cut(scores, actual)
    calibrated_rate = float(np.mean(np.asarray(flags, dtype=bool)))
    flags = best_flags

    predicted, covered = paint_windows(flags, provenance, size, len(truth))
    painted = paint_scores(scores, provenance, size, len(truth))
    values = as_matrix(evaluation.Stream.iloc[0])
    x = np.arange(len(truth))

    fig, (top, middle, bottom) = plt.subplots(
        3, 1, figsize=(14, 7), sharex=True,
        gridspec_kw={'height_ratios': [2.0, 1.6, 0.7], 'hspace': 0.15})

    order = np.argsort(values.std(axis=0))[::-1][:3]
    for offset, channel in enumerate(order):
        column = values[:, channel].astype(float)
        spread = column.std() or 1.0
        top.plot(x, (column - column.mean()) / spread + offset * 4.5, lw=0.6, color='#334155')
    top.set_ylabel('3 most variable' + NEWLINE + 'channels', fontsize=8)
    top.set_yticks([])

    middle.plot(x, painted, lw=0.9, color='#1d4ed8', label='window score')
    middle.axhline(best_cut, color='crimson', ls='--', lw=1.1,
                   label='F1-optimal cut')
    middle.set_ylabel('score', fontsize=8)
    middle.legend(fontsize=7.5, loc='upper left')

    for lo, hi in segments(truth):
        for ax in (top, middle, bottom):
            ax.axvspan(lo, hi, color='#dc2626', alpha=0.28, lw=0)

    bottom.fill_between(x, 1.12, 1.88, where=truth, step='mid', color='#dc2626', lw=0)
    bottom.fill_between(x, 0.12, 0.88, where=predicted, step='mid', color='#1d4ed8', lw=0)
    bottom.fill_between(x, 0.02, 0.10, where=~covered, step='mid', color='#cbd5e1', lw=0)
    bottom.text(-len(truth) * 0.008, 1.5, 'labelled  {0:.1%}'.format(float(truth.mean())),
                fontsize=8, ha='right', va='center', color='#dc2626', fontweight='bold')
    bottom.text(-len(truth) * 0.008, 0.5, 'flagged  {0:.1%}'.format(float(predicted.mean())),
                fontsize=8, ha='right', va='center', color='#1d4ed8')
    bottom.set_ylim(0, 2)
    bottom.set_yticks([])
    bottom.set_xlim(0, len(truth))
    bottom.set_xlabel('test-series point index'
                      '   (grey strip: never covered by a sampled window)')

    for ax in (top, middle, bottom):
        ax.spines[['top', 'right']].set_visible(False)
    top.text(0.0, 1.03, '{0}   (W={1})'.format(stem, size),
             transform=top.transAxes, ha='left', va='bottom', fontsize=11)
    plt.tight_layout()

    FIGURES = PROJECT / 'figures'
    FIGURES.mkdir(parents=True, exist_ok=True)
    out = FIGURES / 'timeeval_marks_{0}.png'.format(stem.replace('.', '_'))
    try:
        fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    except OSError:
        print('  {0} was locked; not rewritten'.format(out.name))
    plt.show()

    inside = painted[truth & np.isfinite(painted)]
    outside = painted[(~truth) & np.isfinite(painted)]
    from sklearn.metrics import precision_recall_fscore_support

    def quality(mask):
        p, r, f, _ = precision_recall_fscore_support(
            np.asarray(actual, dtype=bool), np.asarray(mask, dtype=bool),
            average='binary', zero_division=0)
        return float(p), float(r), float(f)

    f1_quality, youden_quality = quality(best_flags), quality(youden_flags)
    return {'dataset': stem, 'roc_auc': _auc_of.get(stem),
            'calibrated_flagged': calibrated_rate,
            'f1_cut': best_cut, 'f1_flagged': float(np.mean(best_flags)),
            'f1_precision': f1_quality[0], 'f1_recall': f1_quality[1],
            'f1_score': f1_quality[2],
            'youden_cut': youden, 'youden_flagged': float(np.mean(youden_flags)),
            'youden_f1': youden_quality[2],
            'flagged': float(predicted.mean()), 'labelled': float(truth.mean()),
            'covered': float(covered.mean()),
            'median_score_anomalous': float(np.median(inside)) if inside.size else float('nan'),
            'median_score_normal': float(np.median(outside)) if outside.size else float('nan'),
            'file': out.name}


mark_rows = []
for stem in MARK_DATASETS:
    print(stem)
    mark_rows.append(plot_timeeval_marks(stem))
    gc.collect()

marks = pd.DataFrame(mark_rows)
marks['score_ratio'] = marks.median_score_anomalous / marks.median_score_normal
display(marks.style.format({
    'roc_auc': '{:.3f}', 'calibrated_flagged': '{:.1%}',
    'f1_cut': '{:.3g}', 'f1_flagged': '{:.1%}', 'f1_precision': '{:.3f}',
    'f1_recall': '{:.3f}', 'f1_score': '{:.3f}',
    'youden_cut': '{:.3g}', 'youden_flagged': '{:.1%}', 'youden_f1': '{:.3f}',
    'flagged': '{:.1%}', 'labelled': '{:.1%}', 'covered': '{:.1%}',
    'median_score_anomalous': '{:.3g}', 'median_score_normal': '{:.3g}',
    'score_ratio': '{:.2f}'}, na_rep='-').hide(axis='index')
    .set_caption('cuts chosen against the labels - oracle operating points, not achievable '
                 'ones; calibrated_flagged is what the unsupervised threshold produced'))


### 4.6 Accuracy and cost across the experiments

In [ ]:
%%time


def experiment_accuracy():
    """AUC, F1 and adjusted F1 for whichever of the four experiments have results on disk."""
    rows = []

    exp1_result = load_result('experiment1')
    if isinstance(exp1_result, dict):
        ranking = exp1_result.get('ranking') or {}
        throughput = exp1_result.get('throughput') or {}
        rows.append({
            'experiment': 'experiment 1 (bagged widest-first)',
            'roc_auc': ranking.get('roc_auc'), 'pr_auc': ranking.get('pr_auc'),
            'pr_auc_lift': ranking.get('pr_auc_lift'),
            'precision': exp1_result.get('precision'), 'recall': exp1_result.get('recall'),
            'f1': exp1_result.get('f1'), 'adj_f1': exp1_result.get('adjusted_f1'),
            'flagged_rate': throughput.get('flagged_fraction'),
            'seconds': throughput.get('seconds'),
            'units_per_second': throughput.get('points_per_second'),
            'unit': 'points'})

    for name, label in (('experiment2', 'experiment 2 (UMAP windows)'),
                        ('experiment3', 'experiment 3 (bagged windows)'),
                        ('timeeval_jitter', 'timeeval jitter windows')):
        frame = load_result(name)
        if not isinstance(frame, pd.DataFrame) or frame.empty:
            continue
        if 'dataset' in frame.columns and frame.dataset.nunique() > 1:
            datasets = frame.dataset.nunique()
            frame = frame.select_dtypes('number').groupby('window', as_index=False).mean()
            frame['n_datasets'] = datasets
        best = (frame.loc[frame.roc_auc.idxmax()] if 'roc_auc' in frame
                and frame.roc_auc.notna().any() else frame.iloc[0])
        rows.append({
            'experiment': label,
            'roc_auc': best.get('roc_auc'), 'pr_auc': best.get('pr_auc'),
            'pr_auc_lift': best.get('pr_auc_lift'),
            'precision': best.get('precision'), 'recall': best.get('recall'),
            'f1': best.get('f1'), 'adj_f1': best.get('adj_f1'),
            'flagged_rate': best.get('flagged_rate'), 'seconds': best.get('seconds'),
            'units_per_second': best.get('windows_per_second'),
            'unit': 'windows', 'best_window': best.get('window')})

    return pd.DataFrame(rows)


accuracy = experiment_accuracy()
if not accuracy.empty:
    display(accuracy.style.format({
        'roc_auc': '{:.4f}', 'pr_auc': '{:.4f}', 'pr_auc_lift': '{:.2f}',
        'precision': '{:.4f}', 'recall': '{:.4f}', 'f1': '{:.4f}', 'adj_f1': '{:.4f}',
        'flagged_rate': '{:.2%}', 'seconds': '{:,.0f}', 'units_per_second': '{:,.1f}'},
        na_rep='-').hide(axis='index').set_caption(
            'accuracy per experiment; F1 uses each experiment own calibrated threshold'))
    save_result('experiment_accuracy', accuracy)

persisted = (pd.read_csv(STAGE_ROWS_CSV) if STAGE_ROWS_CSV.exists() else pd.DataFrame())
live = pd.DataFrame(STAGE_ROWS)
if not live.empty and not persisted.empty:
    live = live[~live.experiment.isin(set(persisted.experiment))]
all_rows = pd.concat([persisted, live], ignore_index=True) if not (
    persisted.empty and live.empty) else pd.DataFrame()

breakdown = stage_breakdown(all_rows.to_dict('records') if not all_rows.empty else [])
totals = stage_totals(all_rows.to_dict('records') if not all_rows.empty else [])

legacy = EXATHLON_RESULTS / 'stage_breakdown.exp1.csv'
if legacy.exists():
    earlier = pd.read_csv(legacy)
    have = set(breakdown.experiment) if not breakdown.empty else set()
    earlier = earlier[~earlier.experiment.isin(have)]
    if not earlier.empty:
        breakdown = (pd.concat([breakdown, earlier], ignore_index=True)
                     if not breakdown.empty else earlier)
        totals_extra = earlier.groupby('experiment').apply(
            lambda g: pd.Series({
                'stages_recorded': int(g.calls.sum()),
                'stage_seconds': float(g.exclusive_seconds.sum()),
                'peak_rss_mb': float(g.peak_rss_mb.max()),
                'peak_stage': g.loc[g.peak_rss_mb.idxmax(), 'stage']
                if g.peak_rss_mb.notna().any() else None}),
            include_groups=False).reset_index()
        totals = (pd.concat([totals, totals_extra], ignore_index=True)
                  if not totals.empty else totals_extra)

if breakdown.empty:
    print('No stage records: run 7.2, 7.4, 7.5 and 8.3 in this kernel to populate them.')
else:
    display(totals.style.format({
        'stage_seconds': '{:,.0f}', 'peak_rss_mb': '{:,.0f}'}).hide(axis='index')
        .set_caption('totals per experiment; peak_stage is the stage that set the high mark'))

    display(breakdown.style.format({
        'seconds': '{:,.1f}', 'exclusive_seconds': '{:,.1f}', 'peak_rss_mb': '{:,.0f}',
        'mean_delta_mb': '{:,.1f}', 'share_of_time': '{:.1%}'}, na_rep='-')
        .hide(axis='index').set_caption(
            'per stage: exclusive_seconds excludes nested stages; peak_rss_mb is the largest '
            'resident set while that stage was open'))
    save_result('stage_breakdown', breakdown)

    experiments = list(breakdown.experiment.unique())
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6 + 0.15 * len(experiments)))
    stages = list(breakdown.stage.unique())
    palette = plt.get_cmap('tab20')(np.linspace(0, 1, max(len(stages), 2)))
    colour = dict(zip(stages, palette))

    for ax, column, title in (
            (axes[0], 'exclusive_seconds', 'time by stage (seconds, nested time excluded)'),
            (axes[1], 'peak_rss_mb', 'peak resident memory by stage (MB)')):
        base = np.zeros(len(experiments))
        for stage in stages:
            values = np.array([
                float(breakdown[(breakdown.experiment == e) & (breakdown.stage == stage)]
                      [column].sum() or 0.0) for e in experiments])
            if not values.any():
                continue
            if column == 'peak_rss_mb':
                ax.barh(np.arange(len(experiments)) + 0.4 * (stages.index(stage) % 2 - 0.5),
                        values, height=0.35, color=colour[stage], label=stage)
            else:
                ax.barh(np.arange(len(experiments)), values, left=base,
                        color=colour[stage], label=stage)
                base += values
        ax.set_yticks(np.arange(len(experiments)))
        ax.set_yticklabels(experiments, fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.grid(axis='x', color='#e2e8f0')
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, fontsize=7.5, ncol=min(len(labels), 6),
               loc='lower center', frameon=False, bbox_to_anchor=(0.5, -0.10))
    fig.suptitle('Exathlon: where the time and the memory went', y=1.02)
    plt.tight_layout()

    FIGURES = PROJECT / 'figures'
    FIGURES.mkdir(parents=True, exist_ok=True)
    try:
        fig.savefig(FIGURES / 'exathlon_stage_costs.png', dpi=200, bbox_inches='tight',
                    facecolor='white')
        print('wrote exathlon_stage_costs.png')
    except OSError:
        print('exathlon_stage_costs.png was locked; not rewritten')
    plt.show()
